In [ ]:
#"""
#Phase 4: ML Demand Forecasting + Conformal Prediction
#======================================================
#Study area : ต.แม่นาเรือ (Mae Na Rua), Phayao
#Targets    : NIR_A_m3 (Zone A rainfed), GID_B_m3 (Zone B irrigated)
#Strategy   : Direct multi-step, H = 12 weeks
#Stack      : CatBoost + LightGBM → Ridge meta-learner
#Uncertainty: Conformal Prediction (split conformal, α = 0.10)
#Split      : Train 2020-2022 | Calib 2023 | Test 2024
 
#Run order  :
  #Step 1 → step1_download_mei.py   (download NOAA MEI index)
  #Step 2 → step2_feature_engineering.py
  #Step 3 → step3_train_direct.py
  #Step 4 → step4_conformal.py
  #Step 5 → step5_metrics_shap.py
#"""

['crop', 'week', 'stage', 'Kc']
<StringArray>
['rice', 'corn', 'cassava', 'longan', 'rubber']
Length: 5, dtype: str
   crop  week      stage   Kc
0  rice     1  offseason  0.0
1  rice     2  offseason  0.0
2  rice     3  offseason  0.0
3  rice     4  offseason  0.0
4  rice     5  offseason  0.0


In [1]:
# ─────────────────────────────────────────────────────────────────────────────
#  STEP 1 — Download NOAA MEI v2 Index
# ─────────────────────────────────────────────────────────────────────────────
# Save as: step1_download_mei.py
 
import requests
import pandas as pd
import numpy as np
from io import StringIO
 
def download_mei():
    """
    Download MEI v2 (Multivariate ENSO Index) from NOAA PSL.
    Source: https://psl.noaa.gov/enso/mei/
    Format: wide table — rows = years, columns = bimonthly periods
    Returns: long-format DataFrame with columns [year, month, MEI]
    """
    url = "https://psl.noaa.gov/enso/mei/data/meiv2.data"
    print(f"Downloading MEI v2 from {url} ...")
 
    resp = requests.get(url, timeout=30)
    resp.raise_for_status()
 
    lines = resp.text.strip().split("\n")
 
    # Find header line (first line with year range)
    data_lines = []
    for line in lines:
        stripped = line.strip()
        if stripped == "" or stripped.startswith("MEI"):
            continue
        parts = stripped.split()
        if len(parts) >= 2 and parts[0].isdigit():
            data_lines.append(parts)
 
    # Column names: DJ, JF, FM, MA, AM, MJ, JJ, JA, AS, SO, ON, ND
    # These bimonthly values map to the SECOND month in the pair
    bimonth_labels = ["DJ","JF","FM","MA","AM","MJ","JJ","JA","AS","SO","ON","ND"]
    # Month of the second element in each pair (1-indexed)
    second_month   = [  1,   2,   3,   4,   5,   6,   7,   8,   9,  10,  11,  12]
 
    records = []
    for parts in data_lines:
        year = int(parts[0])
        values = parts[1:]
        for i, val_str in enumerate(values[:12]):
            try:
                mei_val = float(val_str)
                if mei_val == -999.0:
                    mei_val = np.nan
            except ValueError:
                mei_val = np.nan
            records.append({
                "year":  year,
                "month": second_month[i],
                "MEI":   mei_val,
            })
 
    df = pd.DataFrame(records)
    df = df[(df["year"] >= 2018) & (df["year"] <= 2024)].copy()
    df.sort_values(["year","month"], inplace=True)
    df.reset_index(drop=True, inplace=True)
 
    df.to_csv("mei_monthly.csv", index=False)
    print(f"Saved mei_monthly.csv — {len(df)} rows")
    print(df.tail())
    return df
 
 
if __name__ == "__main__":
    download_mei()

Saved mei_monthly.csv — 84 rows
    year  month   MEI
79  2024      8 -0.73
80  2024      9 -0.65
81  2024     10 -0.51
82  2024     11 -0.68
83  2024     12 -0.91


In [2]:
"""
Step 2 — Feature Engineering  (v3 — single climate file)
==========================================================
Input  : water_demand_weekly_dual_zone.csv
         climate_weekly_phayao_2020_2024.csv  ← รวม ERA5 + CHIRPS ไว้แล้ว
         mei_monthly.csv
Output : ml_features_phase4.csv

ไม่ต้องใช้ :
  ET0_weekly_phayao_2020_2024.csv   (ซ้ำกับ climate file)
  CHIRPS_weekly_phayao_2020_2024.csv (ซ้ำกับ climate file)
"""

import pandas as pd
import numpy as np

HORIZON      = 12
LAG_WINDOWS  = [1, 2, 3, 4, 8, 12]
ROLL_WINDOWS = [4, 8]

ZONE_CONFIG = {
    "zone_A": {"target": "NIR_A_m3",  "p_eff_col": "P_eff_upland"},
    "zone_B": {"target": "GIR_B_m3",  "p_eff_col": "P_eff_paddy"},
}


def add_lag_features(df, col, lags):
    for lag in lags:
        df[f"{col}_lag{lag}"] = df[col].shift(lag)
    return df


def add_rolling_features(df, col, windows):
    for w in windows:
        df[f"{col}_roll{w}_mean"] = df[col].shift(1).rolling(w).mean()
        df[f"{col}_roll{w}_std"]  = df[col].shift(1).rolling(w).std()
    return df


def build_feature_matrix():

    # ── Load ──────────────────────────────────────────────────────────────
    demand  = pd.read_csv("water_demand_weekly_dual_zone.csv")
    climate = pd.read_csv("climate_weekly_phayao_2020_2024.csv")
    mei     = pd.read_csv("mei_monthly.csv")

    # ── Date + month ──────────────────────────────────────────────────────
    demand["date"] = pd.to_datetime(
        demand["year"].astype(str) + "-W" +
        demand["week"].astype(str).str.zfill(2) + "-1",
        format="%G-W%V-%u"
    )
    demand["month"] = demand["date"].dt.month

    # ── Merge climate (drop duplicate columns already in demand) ──────────
    overlap = [c for c in climate.columns
               if c in demand.columns and c not in ["year", "week"]]
    demand_clean = demand.drop(columns=overlap, errors="ignore")

    df = demand_clean.merge(climate, on=["year", "week"], how="left")

    # ── Merge MEI ─────────────────────────────────────────────────────────
    df = df.merge(mei[["year", "month", "MEI"]], on=["year", "month"], how="left")
    df["MEI"] = df["MEI"].interpolate(method="linear").bfill().ffill()

    # ── Seasonality encoding ──────────────────────────────────────────────
    df["WoY_sin"] = np.sin(2 * np.pi * df["week"] / 52)
    df["WoY_cos"] = np.cos(2 * np.pi * df["week"] / 52)
    df["MoY_sin"] = np.sin(2 * np.pi * df["month"] / 12)
    df["MoY_cos"] = np.cos(2 * np.pi * df["month"] / 12)

    # ── Season encode (ถ้ามี column season) ──────────────────────────────
    if "season" in df.columns:
        season_map = {"dry": 0, "wet": 1, "cool": 2,
                      "summer": 0, "rainy": 1, "winter": 2}
        df["season_enc"] = df["season"].map(season_map)
        if df["season_enc"].isna().any():
            df["season_enc"] = pd.factorize(df["season"])[0]

    print(f"Merged: {df.shape[0]} rows x {df.shape[1]} cols")
    nan_counts = df.isna().sum()
    if nan_counts.any():
        print(f"NaN columns:\n{nan_counts[nan_counts > 0]}\n")

    # ── Per-zone feature matrices ─────────────────────────────────────────
    frames = []

    for zone_label, cfg in ZONE_CONFIG.items():
        target_col = cfg["target"]
        p_eff_col  = cfg["p_eff_col"]

        wanted = [
            "year", "week", "month", "date",
            "ET0_mm_week", "T_mean", "RH_pct", "VPD_kPa", "u2_ms", "Rn_MJ",
            "P_mm_week", p_eff_col,
            "SPI_4", "drought_flag", "AI_week",
            "MEI", "WoY_sin", "WoY_cos", "MoY_sin", "MoY_cos",
            target_col,
        ]
        if "season_enc" in df.columns:
            wanted.append("season_enc")

        cols = [c for c in wanted if c in df.columns]
        z = df[cols].copy().sort_values(["year", "week"]).reset_index(drop=True)

        z = z.rename(columns={p_eff_col: "P_eff_mm"})

        # ── Lag + rolling on target ───────────────────────────────────────
        z = add_lag_features(z, target_col, LAG_WINDOWS)
        z = add_rolling_features(z, target_col, ROLL_WINDOWS)

        # ── Lag on climate drivers ────────────────────────────────────────
        z = add_lag_features(z, "ET0_mm_week", [1, 2, 4])
        z = add_lag_features(z, "P_mm_week",   [1, 2, 4])
        z = add_lag_features(z, "VPD_kPa",     [1, 2])
        z = add_lag_features(z, "MEI",         [4, 8])

        # ── Direct horizon targets ────────────────────────────────────────
        for h in range(1, HORIZON + 1):
            z[f"y_h{h}"] = z[target_col].shift(-h)

        z["zone"]       = zone_label
        z["target_col"] = target_col
        frames.append(z)

    # ── Combine ───────────────────────────────────────────────────────────
    ml = pd.concat(frames, ignore_index=True)
    ml = ml.sort_values(["year", "week", "zone"]).reset_index(drop=True)

    horizon_cols = [f"y_h{h}" for h in range(1, HORIZON + 1)]
    ml = ml.dropna(subset=horizon_cols, how="all")

    ml.to_csv("ml_features_phase4.csv", index=False)

    # ── Summary ───────────────────────────────────────────────────────────
    meta_cols = {"year","week","month","date","zone","target_col",
                 "NIR_A_m3","GIR_B_m3","P_4week"}
    feature_cols = [c for c in ml.columns
                    if c not in meta_cols and not c.startswith("y_h")]

    print(f"\n{'='*55}")
    print(f"Output   : ml_features_phase4.csv")
    print(f"Shape    : {ml.shape[0]} rows x {ml.shape[1]} cols")
    print(f"Zones    : {list(ml['zone'].unique())}")
    print(f"Years    : {sorted(ml['year'].unique())}")
    print(f"Features ({len(feature_cols)}):")
    for i, f in enumerate(feature_cols, 1):
        print(f"  {i:2d}. {f}")
    print(f"Targets  : y_h1 ... y_h{HORIZON}")
    print(f"{'='*55}")
    return ml


if __name__ == "__main__":
    build_feature_matrix()

Merged: 260 rows x 33 cols
NaN columns:
P_4week    3
dtype: int64


Output   : ml_features_phase4.csv
Shape    : 518 rows x 67 cols
Zones    : ['zone_A', 'zone_B']
Years    : [np.int64(2020), np.int64(2021), np.int64(2022), np.int64(2023), np.int64(2024)]
Features (47):
   1. ET0_mm_week
   2. T_mean
   3. RH_pct
   4. VPD_kPa
   5. u2_ms
   6. Rn_MJ
   7. P_mm_week
   8. P_eff_mm
   9. SPI_4
  10. drought_flag
  11. AI_week
  12. MEI
  13. WoY_sin
  14. WoY_cos
  15. MoY_sin
  16. MoY_cos
  17. season_enc
  18. NIR_A_m3_lag1
  19. NIR_A_m3_lag2
  20. NIR_A_m3_lag3
  21. NIR_A_m3_lag4
  22. NIR_A_m3_lag8
  23. NIR_A_m3_lag12
  24. NIR_A_m3_roll4_mean
  25. NIR_A_m3_roll4_std
  26. NIR_A_m3_roll8_mean
  27. NIR_A_m3_roll8_std
  28. ET0_mm_week_lag1
  29. ET0_mm_week_lag2
  30. ET0_mm_week_lag4
  31. P_mm_week_lag1
  32. P_mm_week_lag2
  33. P_mm_week_lag4
  34. VPD_kPa_lag1
  35. VPD_kPa_lag2
  36. MEI_lag4
  37. MEI_lag8
  38. GIR_B_m3_lag1
  39. GIR_B_m3_lag2
  40. GIR_B_m3_lag3
  41.

In [ ]:
"""
Step 3a — CatBoost Rerun รอบ 3 (zone A h=2, 7, 8 เท่านั้น)
=============================================================
Input  : ml_features_phase4.csv
         catboost_models.pkl        ← รอบ 2 (จะ overwrite เฉพาะ 3 keys)
Output : catboost_models.pkl        ← merged (รอบ 2 + รอบ 3)
         catboost_tuning_log_r3.csv

เปลี่ยนจากรอบ 2:
  - warmstart จาก best params รอบ 2 (trial แรกเริ่มจากจุดที่รู้แล้ว)
  - depth min → 6  (ตัด depth=5 ออกสำหรับ zone A)
  - learning_rate min → 0.15  (ตัด lr ต่ำที่ทำให้ h=2 แย่)
  - l2_leaf_reg max → 6.0  (เข้มกว่ารอบ 2)
  - N_TRIALS → 120
  - EARLY_STOP → 80
"""

import pandas as pd
import numpy as np
import optuna
import joblib
import warnings
warnings.filterwarnings("ignore")
optuna.logging.set_verbosity(optuna.logging.WARNING)

from catboost import CatBoostRegressor
from sklearn.metrics import mean_absolute_error

# ── Config ────────────────────────────────────────────────────────────────────
HORIZON     = 12
TRAIN_YEARS = [2020, 2021, 2022]
CALIB_YEAR  = 2023
TEST_YEAR   = 2024
N_TRIALS    = 120
EARLY_STOP  = 80

# rerun เฉพาะ zone A horizons เหล่านี้
RERUN = [("zone_A", 2), ("zone_A", 7), ("zone_A", 8)]

# best params จากรอบ 2 — เป็น starting point ของ trial แรก
WARMSTART = {
    ("zone_A", 2): {
        "iterations":        1372,
        "depth":             7,
        "learning_rate":     0.1229,
        "l2_leaf_reg":       3.366,
        "subsample":         0.912,
        "colsample_bylevel": 0.691,
        "min_data_in_leaf":  15,
    },
    ("zone_A", 7): {
        "iterations":        867,
        "depth":             8,
        "learning_rate":     0.2379,
        "l2_leaf_reg":       6.0,    # clamp ให้อยู่ใน range ใหม่
        "subsample":         0.914,
        "colsample_bylevel": 0.513,
        "min_data_in_leaf":  11,
    },
    ("zone_A", 8): {
        "iterations":        1392,
        "depth":             6,      # เพิ่มจาก 5 → 6 (min ใหม่)
        "learning_rate":     0.1544,
        "l2_leaf_reg":       1.004,
        "subsample":         0.865,
        "colsample_bylevel": 0.615,
        "min_data_in_leaf":  6,
    },
}


def get_feature_cols(df: pd.DataFrame, df_zone: pd.DataFrame = None) -> list:
    exclude = {"year", "week", "month", "date", "zone", "target_col",
               "NIR_A_m3", "GIR_B_m3", "P_4week"}
    exclude |= {f"y_h{h}" for h in range(1, HORIZON + 1)}
    cols = [c for c in df.columns if c not in exclude]
    if df_zone is not None:
        cols = [c for c in cols if not df_zone[c].isna().all()]
    return cols


def tune_catboost(X_train, y_train, X_calib, y_calib,
                  warmstart_params: dict = None) -> tuple:
    def objective(trial):
        params = {
            "iterations":        trial.suggest_int("iterations", 700, 2000),
            "depth":             trial.suggest_int("depth", 6, 10),
            "learning_rate":     trial.suggest_float("learning_rate", 0.15, 0.30, log=True),
            "l2_leaf_reg":       trial.suggest_float("l2_leaf_reg", 1.0, 6.0),
            "subsample":         trial.suggest_float("subsample", 0.6, 1.0),
            "colsample_bylevel": trial.suggest_float("colsample_bylevel", 0.5, 1.0),
            "min_data_in_leaf":  trial.suggest_int("min_data_in_leaf", 2, 20),
            "verbose":      False,
            "random_seed":  42,
        }
        m = CatBoostRegressor(**params)
        m.fit(X_train, y_train,
              eval_set=(X_calib, y_calib),
              early_stopping_rounds=EARLY_STOP)
        return mean_absolute_error(y_calib, m.predict(X_calib))

    study = optuna.create_study(
        direction="minimize",
        sampler=optuna.samplers.TPESampler(seed=42),
    )
    if warmstart_params is not None:
        # clamp warmstart values ให้อยู่ใน range ใหม่ก่อน enqueue
        ws = dict(warmstart_params)
        ws["depth"]         = max(6,    min(10,  ws["depth"]))
        ws["learning_rate"] = max(0.15, min(0.30, ws["learning_rate"]))
        ws["l2_leaf_reg"]   = max(1.0,  min(6.0,  ws["l2_leaf_reg"]))
        study.enqueue_trial(ws)

    study.optimize(objective, n_trials=N_TRIALS, show_progress_bar=True)

    best_params = study.best_params
    best_model  = CatBoostRegressor(**best_params, verbose=False, random_seed=42)
    best_model.fit(X_train, y_train)
    best_mae = mean_absolute_error(y_calib, best_model.predict(X_calib))

    return best_model, best_params, best_mae


def main():
    df = pd.read_csv("ml_features_phase4.csv")

    # MAE รอบ 2 สำหรับ compare
    r2_mae = {
        ("zone_A", 2): 93966.80,
        ("zone_A", 7): 84138.09,
        ("zone_A", 8): 85954.44,
    }

    # โหลด models รอบ 2
    try:
        all_models = joblib.load("catboost_models.pkl")
        print(f"Loaded catboost_models.pkl ({len(all_models)} models)")
    except FileNotFoundError:
        print("catboost_models.pkl not found — starting fresh dict")
        all_models = {}

    new_models = {}
    log_rows   = []

    for zone, h in RERUN:
        df_zone      = df[df["zone"] == zone].copy()
        feature_cols = get_feature_cols(df, df_zone)

        target = f"y_h{h}"
        valid  = df_zone.dropna(subset=feature_cols + [target])

        train = valid[valid["year"].isin(TRAIN_YEARS)]
        calib = valid[valid["year"] == CALIB_YEAR]

        X_train = train[feature_cols].values
        y_train = train[target].values
        X_calib = calib[feature_cols].values
        y_calib = calib[target].values

        print(f"\n{'='*60}")
        print(f"[{zone}] h={h:02d}  |  train={len(X_train)}  calib={len(X_calib)}")
        print(f"  รอบ 2 MAE = {r2_mae[(zone,h)]:,.0f} m³  ← target to beat")
        print(f"{'='*60}")

        ws    = WARMSTART.get((zone, h))
        model, params, mae = tune_catboost(
            X_train, y_train, X_calib, y_calib,
            warmstart_params=ws,
        )
        new_models[(zone, h)] = model

        r2     = r2_mae[(zone, h)]
        delta  = mae - r2
        symbol = "✅" if delta < 0 else "❌"
        print(f"\n  {symbol} รอบ 3 MAE = {mae:,.2f}  |  delta = {delta:+,.0f} m³")
        print(f"  best params: {params}")

        log_rows.append({
            "zone":             zone,
            "horizon":          h,
            "calib_mae_r2":     r2,
            "calib_mae_r3":     round(mae, 2),
            "delta":            round(delta, 2),
            **{f"cat_{k}": v for k, v in params.items()},
        })

    # ── Merge: overwrite เฉพาะ 3 keys ────────────────────────────────────
    for key, model in new_models.items():
        # เก็บเฉพาะถ้า MAE รอบ 3 ดีกว่ารอบ 2
        zone, h = key
        r3_mae = log_rows[[r["horizon"] == h and r["zone"] == zone
                            for r in log_rows].index(True)]["calib_mae_r3"]
        if r3_mae < r2_mae[key]:
            all_models[key] = model
            print(f"\n[merge] ({zone}, h={h}) → replaced (MAE {r2_mae[key]:,.0f} → {r3_mae:,.0f})")
        else:
            print(f"\n[merge] ({zone}, h={h}) → kept รอบ 2 (รอบ 3 ไม่ดีกว่า)")

    joblib.dump(all_models, "catboost_models.pkl")
    print(f"\nSaved catboost_models.pkl ({len(all_models)} models)")

    # ── Summary ───────────────────────────────────────────────────────────
    log_df = pd.DataFrame(log_rows)
    log_df.to_csv("catboost_tuning_log_r3.csv", index=False)

    print("\n" + "="*55)
    print("Round 3 summary")
    print("="*55)
    print(log_df[["zone","horizon","calib_mae_r2","calib_mae_r3","delta"]]
          .to_string(index=False))


if __name__ == "__main__":
    main()

Loaded catboost_models.pkl (24 models)

[zone_A] h=02  |  train=144  calib=52
  รอบ 2 MAE = 93,967 m³  ← target to beat


  0%|          | 0/120 [00:00<?, ?it/s]


  ✅ รอบ 3 MAE = 88,273.18  |  delta = -5,694 m³
  best params: {'iterations': 1391, 'depth': 10, 'learning_rate': 0.1827812924371775, 'l2_leaf_reg': 1.1151056168415359, 'subsample': 0.8284719721523301, 'colsample_bylevel': 0.6232043098962213, 'min_data_in_leaf': 10}

[zone_A] h=07  |  train=144  calib=52
  รอบ 2 MAE = 84,138 m³  ← target to beat


  0%|          | 0/120 [00:00<?, ?it/s]


  ❌ รอบ 3 MAE = 87,178.12  |  delta = +3,040 m³
  best params: {'iterations': 1458, 'depth': 9, 'learning_rate': 0.26994425277880535, 'l2_leaf_reg': 4.560274822924958, 'subsample': 0.8544932325143816, 'colsample_bylevel': 0.908408107340333, 'min_data_in_leaf': 12}

[zone_A] h=08  |  train=144  calib=52
  รอบ 2 MAE = 85,954 m³  ← target to beat


  0%|          | 0/120 [00:00<?, ?it/s]


  ✅ รอบ 3 MAE = 84,788.43  |  delta = -1,166 m³
  best params: {'iterations': 1170, 'depth': 8, 'learning_rate': 0.21133633213323585, 'l2_leaf_reg': 3.4734916988109106, 'subsample': 0.8089851580765991, 'colsample_bylevel': 0.9140089621926575, 'min_data_in_leaf': 7}

[merge] (zone_A, h=2) → replaced (MAE 93,967 → 88,273)

[merge] (zone_A, h=7) → kept รอบ 2 (รอบ 3 ไม่ดีกว่า)

[merge] (zone_A, h=8) → replaced (MAE 85,954 → 84,788)

Saved catboost_models.pkl (24 models)

Round 3 summary
  zone  horizon  calib_mae_r2  calib_mae_r3    delta
zone_A        2      93966.80      88273.18 -5693.62
zone_A        7      84138.09      87178.12  3040.03
zone_A        8      85954.44      84788.43 -1166.01


In [14]:
"""
Step 3b — LightGBM Training + Optuna Tuning
============================================
Input  : ml_features_phase4.csv
Output : lightgbm_models.pkl
         lightgbm_tuning_log.csv

Run    : python step3b_lightgbm.py
"""

import pandas as pd
import numpy as np
import optuna
import joblib
import warnings
warnings.filterwarnings("ignore")
optuna.logging.set_verbosity(optuna.logging.WARNING)

import lightgbm as lgb
from sklearn.metrics import mean_absolute_error

# ── Config ────────────────────────────────────────────────────────────────────
HORIZON     = 12
ZONES       = ["zone_A", "zone_B"]
TRAIN_YEARS = [2020, 2021, 2022]
CALIB_YEAR  = 2023
N_TRIALS    = 80        # เพิ่มเป็น 150+ สำหรับ final run
EARLY_STOP  = 50


def get_feature_cols(df: pd.DataFrame, df_zone: pd.DataFrame = None) -> list:
    exclude = {"year", "week", "month", "date", "zone", "target_col",
               "NIR_A_m3", "GID_B_m3", "P_4week"}
    exclude |= {f"y_h{h}" for h in range(1, HORIZON + 1)}
    cols = [c for c in df.columns if c not in exclude]
    if df_zone is not None:
        cols = [c for c in cols if not df_zone[c].isna().all()]
    return cols


def tune_lightgbm(X_train, y_train, X_calib, y_calib) -> tuple:
    """
    Bayesian search over LightGBM hyperparameters.
    Returns (best_model, best_params, best_mae).
    """
    def objective(trial):
        params = {
            "objective":        "regression",
            "metric":           "mae",
            "verbosity":        -1,
            "boosting_type":    "gbdt",
            "n_estimators":     trial.suggest_int("n_estimators", 600, 2000),
            "learning_rate":    trial.suggest_float("learning_rate", 0.1, 0.3, log=True),
            "num_leaves":       trial.suggest_int("num_leaves", 20, 120),
            "max_depth":        trial.suggest_int("max_depth", 4, 12),
            "min_child_samples":trial.suggest_int("min_child_samples", 5, 35),
            "subsample":        trial.suggest_float("subsample", 0.5, 1.0),
            "subsample_freq":   1,
            "colsample_bytree": trial.suggest_float("colsample_bytree", 0.4, 1.0),
            "reg_alpha":        trial.suggest_float("reg_alpha", 1e-4, 3.0, log=True),
            "reg_lambda":       trial.suggest_float("reg_lambda", 1e-4, 3.0, log=True),
        }
        m = lgb.LGBMRegressor(**params)
        m.fit(
            X_train, y_train,
            eval_set=[(X_calib, y_calib)],
            callbacks=[lgb.early_stopping(EARLY_STOP), lgb.log_evaluation(-1)],
        )
        return mean_absolute_error(y_calib, m.predict(X_calib))

    study = optuna.create_study(
        direction="minimize",
        sampler=optuna.samplers.TPESampler(seed=42),
    )
    study.optimize(objective, n_trials=N_TRIALS, show_progress_bar=True)

    best_params = {
        "objective": "regression", "metric": "mae",
        "verbosity": -1, **study.best_params,
    }
    best_model = lgb.LGBMRegressor(**best_params)
    best_model.fit(
        X_train, y_train,
        eval_set=[(X_calib, y_calib)],
        callbacks=[lgb.early_stopping(EARLY_STOP), lgb.log_evaluation(-1)],
    )
    best_mae = mean_absolute_error(y_calib, best_model.predict(X_calib))

    return best_model, study.best_params, best_mae


def main():
    df = pd.read_csv("ml_features_phase4.csv")

    lgb_models = {}   # key: (zone, h)  value: fitted LGBMRegressor
    log_rows   = []

    for zone in ZONES:
        df_zone = df[df["zone"] == zone].copy()
        feature_cols = get_feature_cols(df, df_zone)
        print(f"\n{'='*60}")
        print(f"Zone: {zone}")
        print(f"{'='*60}")

        for h in range(1, HORIZON + 1):
            target = f"y_h{h}"
            valid  = df_zone.dropna(subset=feature_cols + [target])

            train = valid[valid["year"].isin(TRAIN_YEARS)]
            calib = valid[valid["year"] == CALIB_YEAR]

            X_train = train[feature_cols].values
            y_train = train[target].values
            X_calib = calib[feature_cols].values
            y_calib = calib[target].values

            print(f"\n  h={h:02d} | train={len(X_train)} calib={len(X_calib)}")

            model, params, mae = tune_lightgbm(X_train, y_train, X_calib, y_calib)
            lgb_models[(zone, h)] = model

            log_rows.append({
                "zone": zone, "horizon": h, "calib_mae": round(mae, 2),
                **{f"lgb_{k}": v for k, v in params.items()},
            })
            print(f"  → Best MAE = {mae:.2f} m³  |  params: {params}")

    # ── Save ──────────────────────────────────────────────────────────────
    joblib.dump(lgb_models, "lightgbm_models.pkl")
    pd.DataFrame(log_rows).to_csv("lightgbm_tuning_log.csv", index=False)
    print(f"\nSaved lightgbm_models.pkl  ({len(lgb_models)} models)")
    print("Saved lightgbm_tuning_log.csv")


if __name__ == "__main__":
    main()


Zone: zone_A

  h=01 | train=144 calib=52


  0%|          | 0/80 [00:00<?, ?it/s]

Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[1]	valid_0's l1: 81473.1
Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[29]	valid_0's l1: 82479.6
Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[1]	valid_0's l1: 83343
Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[10]	valid_0's l1: 81149.5
Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[9]	valid_0's l1: 81905.6
Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[5]	valid_0's l1: 84653.7
Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[1]	valid_0's l1: 82062.5
Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[1]	valid_0's l1: 82681.3
Training until validatio

  0%|          | 0/80 [00:00<?, ?it/s]

Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[4]	valid_0's l1: 81700.9
Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[3]	valid_0's l1: 82361.7
Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[10]	valid_0's l1: 83967.5
Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[4]	valid_0's l1: 78148.9
Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[10]	valid_0's l1: 84237.8
Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[1]	valid_0's l1: 83564
Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[1]	valid_0's l1: 79630.9
Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[1]	valid_0's l1: 84602.5
Training until validatio

...

Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[28]	valid_0's l1: 26821.9
Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[69]	valid_0's l1: 28905
Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[13]	valid_0's l1: 28365.7
Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[17]	valid_0's l1: 28133.5
Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[73]	valid_0's l1: 27416.4
Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[19]	valid_0's l1: 27200.9
Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[10]	valid_0's l1: 27858.5
Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[84]	valid_0's l1: 29195.1
Training until val

  0%|          | 0/80 [00:00<?, ?it/s]

Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[2]	valid_0's l1: 30135.4
Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[13]	valid_0's l1: 29906.8
Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[9]	valid_0's l1: 30231.4
Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[4]	valid_0's l1: 29141.1
Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[18]	valid_0's l1: 30489.9
Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[1]	valid_0's l1: 31226.8
Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[10]	valid_0's l1: 29064.1
Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[21]	valid_0's l1: 28991.5
Training until valid

  0%|          | 0/80 [00:00<?, ?it/s]

Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[6]	valid_0's l1: 27939.5
Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[64]	valid_0's l1: 28136.4
Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[68]	valid_0's l1: 27220.2
Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[13]	valid_0's l1: 27433.1
Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[25]	valid_0's l1: 28654.3
Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[11]	valid_0's l1: 28496.6
Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[3]	valid_0's l1: 29317.7
Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[36]	valid_0's l1: 27709.1
Training until val

In [15]:
"""
Step 3b — LightGBM Rerun รอบ 3
================================
Zone A  : rerun h=2,5,6,8,11,12  (horizons ที่แย่ลงในรอบ 2)
Zone B  : rerun ทั้ง 12 horizons  (8/12 แย่ลง → คืน search space กว้าง)

Search space แยกต่าม zone:
  Zone A : lr 0.10–0.30, depth 4–12, reg 1e-4–3.0
  Zone B : lr 0.05–0.30, depth 3–12, reg_alpha 1e-4–8.0, reg_lambda 1e-4–5.0

Warmstart: best params จากรอบที่ดีที่สุด (round 1 หรือ 2) ต่อ horizon
Merge    : auto-keep ผลที่ดีที่สุดจากทุก round ก่อน save

Input  : ml_features_phase4.csv
         lightgbm_models.pkl   ← รอบ 2
Output : lightgbm_models.pkl   ← merged best
         lightgbm_tuning_log_r3.csv
"""

import pandas as pd
import numpy as np
import optuna
import joblib
import warnings
warnings.filterwarnings("ignore")
optuna.logging.set_verbosity(optuna.logging.WARNING)

import lightgbm as lgb
from sklearn.metrics import mean_absolute_error

# ── Config ────────────────────────────────────────────────────────────────────
HORIZON     = 12
TRAIN_YEARS = [2020, 2021, 2022]
CALIB_YEAR  = 2023
N_TRIALS    = 120
EARLY_STOP  = 80

# Zone A: rerun horizons ที่แย่ลงในรอบ 2
RERUN_A = [2, 5, 6, 8, 11, 12]
# Zone B: rerun ทั้งหมด (search space รอบ 2 ทำให้แย่ลง 8/12)
RERUN_B = list(range(1, 13))

# Search space แยก zone
SEARCH_SPACE = {
    "zone_A": dict(
        lr_min=0.10,  lr_max=0.30,
        depth_min=4,  depth_max=12,
        alpha_min=1e-4, alpha_max=3.0,
        lambda_min=1e-4, lambda_max=3.0,
    ),
    "zone_B": dict(
        lr_min=0.05,  lr_max=0.30,   # คืน lr ต่ำ
        depth_min=3,  depth_max=12,  # คืน depth=3
        alpha_min=1e-4, alpha_max=8.0,  # zone B ใช้ alpha สูงได้
        lambda_min=1e-4, lambda_max=5.0,
    ),
}

# ── Warmstart: best params จาก round ที่ดีกว่า ───────────────────────────────
# Zone A — warmstart จากรอบ 1 สำหรับ horizon ที่รอบ 1 ดีกว่า
# Zone B — warmstart จากรอบ 1 (ดีกว่ารอบ 2 ใน 8/12 horizons)

WARMSTART = {
    # Zone A — ใช้ round 1 params สำหรับ h ที่ round 1 ดีกว่า
    ("zone_A", 2):  {"n_estimators":613,  "learning_rate":0.2908, "num_leaves":133,
                     "max_depth":6,  "min_child_samples":6,  "subsample":0.6447,
                     "colsample_bytree":0.5710, "reg_alpha":0.0137, "reg_lambda":0.3997},
    ("zone_A", 5):  {"n_estimators":987,  "learning_rate":0.2537, "num_leaves":65,
                     "max_depth":6,  "min_child_samples":10, "subsample":0.5510,
                     "colsample_bytree":0.7389, "reg_alpha":0.0003, "reg_lambda":1.5321},
    ("zone_A", 6):  {"n_estimators":1460, "learning_rate":0.0561, "num_leaves":20,
                     "max_depth":8,  "min_child_samples":5,  "subsample":0.5338,
                     "colsample_bytree":0.8334, "reg_alpha":0.0038, "reg_lambda":9.984},
    ("zone_A", 8):  {"n_estimators":1850, "learning_rate":0.1372, "num_leaves":125,
                     "max_depth":10, "min_child_samples":18, "subsample":0.9707,
                     "colsample_bytree":0.7645, "reg_alpha":0.0001, "reg_lambda":0.0002},
    ("zone_A", 11): {"n_estimators":937,  "learning_rate":0.2452, "num_leaves":115,
                     "max_depth":8,  "min_child_samples":12, "subsample":0.5780,
                     "colsample_bytree":0.4349, "reg_alpha":2.1423, "reg_lambda":0.1013},
    ("zone_A", 12): {"n_estimators":1014, "learning_rate":0.2605, "num_leaves":120,
                     "max_depth":10, "min_child_samples":10, "subsample":0.5020,
                     "colsample_bytree":0.4536, "reg_alpha":0.0176, "reg_lambda":0.2303},
    # Zone B — warmstart จาก round 1 ทุก horizon
    ("zone_B", 1):  {"n_estimators":593,  "learning_rate":0.1944, "num_leaves":137,
                     "max_depth":11, "min_child_samples":15, "subsample":0.5750,
                     "colsample_bytree":0.5244, "reg_alpha":1.7348, "reg_lambda":0.0016},
    ("zone_B", 2):  {"n_estimators":1403, "learning_rate":0.0911, "num_leaves":51,
                     "max_depth":10, "min_child_samples":14, "subsample":0.5042,
                     "colsample_bytree":0.9660, "reg_alpha":0.1845, "reg_lambda":1.4534},
    ("zone_B", 3):  {"n_estimators":724,  "learning_rate":0.1457, "num_leaves":94,
                     "max_depth":11, "min_child_samples":8,  "subsample":0.5824,
                     "colsample_bytree":0.5041, "reg_alpha":0.0089, "reg_lambda":0.0006},
    ("zone_B", 4):  {"n_estimators":1670, "learning_rate":0.0525, "num_leaves":110,
                     "max_depth":12, "min_child_samples":17, "subsample":0.9754,
                     "colsample_bytree":0.9753, "reg_alpha":6.7842, "reg_lambda":0.0172},
    ("zone_B", 5):  {"n_estimators":1030, "learning_rate":0.1644, "num_leaves":23,
                     "max_depth":3,  "min_child_samples":12, "subsample":0.6699,
                     "colsample_bytree":0.9891, "reg_alpha":0.0013, "reg_lambda":2.3709},
    ("zone_B", 6):  {"n_estimators":508,  "learning_rate":0.2167, "num_leaves":76,
                     "max_depth":9,  "min_child_samples":22, "subsample":0.5649,
                     "colsample_bytree":0.5238, "reg_alpha":1.9599, "reg_lambda":0.9322},
    ("zone_B", 7):  {"n_estimators":1501, "learning_rate":0.2966, "num_leaves":30,
                     "max_depth":9,  "min_child_samples":45, "subsample":0.9567,
                     "colsample_bytree":0.7354, "reg_alpha":0.0008, "reg_lambda":0.0085},
    ("zone_B", 8):  {"n_estimators":1992, "learning_rate":0.1216, "num_leaves":46,
                     "max_depth":12, "min_child_samples":8,  "subsample":0.9992,
                     "colsample_bytree":0.8432, "reg_alpha":0.0154, "reg_lambda":0.1665},
    ("zone_B", 9):  {"n_estimators":1444, "learning_rate":0.1406, "num_leaves":114,
                     "max_depth":6,  "min_child_samples":15, "subsample":0.8574,
                     "colsample_bytree":0.8103, "reg_alpha":0.0455, "reg_lambda":0.0196},
    ("zone_B", 10): {"n_estimators":1025, "learning_rate":0.2855, "num_leaves":150,
                     "max_depth":9,  "min_child_samples":7,  "subsample":0.9562,
                     "colsample_bytree":0.5453, "reg_alpha":0.0039, "reg_lambda":0.0048},
    ("zone_B", 11): {"n_estimators":872,  "learning_rate":0.2981, "num_leaves":61,
                     "max_depth":3,  "min_child_samples":6,  "subsample":0.6941,
                     "colsample_bytree":0.8206, "reg_alpha":0.0047, "reg_lambda":4.9100},
    ("zone_B", 12): {"n_estimators":1054, "learning_rate":0.2669, "num_leaves":47,
                     "max_depth":12, "min_child_samples":15, "subsample":0.5788,
                     "colsample_bytree":0.5464, "reg_alpha":9.6406, "reg_lambda":6.2304},
}

# MAE ที่ดีที่สุดในปัจจุบัน (best of round 1 & 2) — สำหรับ merge check
BEST_MAE = {
    ("zone_A", 1):  81333.74, ("zone_A", 2):  86248.04, ("zone_A", 3):  79964.46,
    ("zone_A", 4):  82513.25, ("zone_A", 5):  81620.34, ("zone_A", 6):  77579.03,
    ("zone_A", 7):  81611.10, ("zone_A", 8):  81030.43, ("zone_A", 9):  75281.45,
    ("zone_A", 10): 80494.66, ("zone_A", 11): 82826.25, ("zone_A", 12): 76874.42,
    ("zone_B", 1):  27175.79, ("zone_B", 2):  29004.31, ("zone_B", 3):  26252.54,
    ("zone_B", 4):  26199.50, ("zone_B", 5):  26861.40, ("zone_B", 6):  25082.02,
    ("zone_B", 7):  26549.14, ("zone_B", 8):  26310.01, ("zone_B", 9):  28117.68,
    ("zone_B", 10): 24883.06, ("zone_B", 11): 29121.29, ("zone_B", 12): 26924.27,
}


def get_feature_cols(df: pd.DataFrame, df_zone: pd.DataFrame = None) -> list:
    exclude = {"year", "week", "month", "date", "zone", "target_col",
               "NIR_A_m3", "GIR_B_m3", "P_4week"}
    exclude |= {f"y_h{h}" for h in range(1, HORIZON + 1)}
    cols = [c for c in df.columns if c not in exclude]
    if df_zone is not None:
        cols = [c for c in cols if not df_zone[c].isna().all()]
    return cols


def tune_lightgbm(X_train, y_train, X_calib, y_calib,
                  zone: str, warmstart_params: dict = None) -> tuple:
    sp = SEARCH_SPACE[zone]

    def objective(trial):
        params = {
            "objective":         "regression",
            "metric":            "mae",
            "verbosity":         -1,
            "boosting_type":     "gbdt",
            "n_estimators":      trial.suggest_int("n_estimators", 600, 2000),
            "learning_rate":     trial.suggest_float("learning_rate",
                                     sp["lr_min"], sp["lr_max"], log=True),
            "num_leaves":        trial.suggest_int("num_leaves", 20, 120),
            "max_depth":         trial.suggest_int("max_depth",
                                     sp["depth_min"], sp["depth_max"]),
            "min_child_samples": trial.suggest_int("min_child_samples", 5, 35),
            "subsample":         trial.suggest_float("subsample", 0.5, 1.0),
            "subsample_freq":    1,
            "colsample_bytree":  trial.suggest_float("colsample_bytree", 0.45, 1.0),
            "reg_alpha":         trial.suggest_float("reg_alpha",
                                     sp["alpha_min"], sp["alpha_max"], log=True),
            "reg_lambda":        trial.suggest_float("reg_lambda",
                                     sp["lambda_min"], sp["lambda_max"], log=True),
        }
        m = lgb.LGBMRegressor(**params)
        m.fit(X_train, y_train,
              eval_set=[(X_calib, y_calib)],
              callbacks=[lgb.early_stopping(EARLY_STOP), lgb.log_evaluation(-1)])
        return mean_absolute_error(y_calib, m.predict(X_calib))

    study = optuna.create_study(
        direction="minimize",
        sampler=optuna.samplers.TPESampler(seed=42),
    )
    if warmstart_params is not None:
        # clamp warmstart ให้อยู่ใน range ใหม่
        ws = dict(warmstart_params)
        ws["learning_rate"]  = max(sp["lr_min"],    min(sp["lr_max"],    ws["learning_rate"]))
        ws["max_depth"]      = max(sp["depth_min"], min(sp["depth_max"], ws["max_depth"]))
        ws["reg_alpha"]      = max(sp["alpha_min"], min(sp["alpha_max"], ws["reg_alpha"]))
        ws["reg_lambda"]     = max(sp["lambda_min"],min(sp["lambda_max"],ws["reg_lambda"]))
        ws["num_leaves"]     = max(20, min(120, ws["num_leaves"]))
        ws["n_estimators"]   = max(600, min(2000, ws["n_estimators"]))
        ws["min_child_samples"] = max(5, min(35, ws["min_child_samples"]))
        study.enqueue_trial(ws)

    study.optimize(objective, n_trials=N_TRIALS, show_progress_bar=True)

    best_params = {
        "objective": "regression", "metric": "mae",
        "verbosity": -1, **study.best_params,
    }
    best_model = lgb.LGBMRegressor(**best_params)
    best_model.fit(X_train, y_train,
                   eval_set=[(X_calib, y_calib)],
                   callbacks=[lgb.early_stopping(EARLY_STOP), lgb.log_evaluation(-1)])
    best_mae = mean_absolute_error(y_calib, best_model.predict(X_calib))

    return best_model, study.best_params, best_mae


def main():
    df = pd.read_csv("ml_features_phase4.csv")

    try:
        all_models = joblib.load("lightgbm_models.pkl")
        print(f"Loaded lightgbm_models.pkl ({len(all_models)} models)")
    except FileNotFoundError:
        print("lightgbm_models.pkl not found — starting fresh dict")
        all_models = {}

    rerun_list = (
        [("zone_A", h) for h in RERUN_A] +
        [("zone_B", h) for h in RERUN_B]
    )
    print(f"\nTotal rerun: {len(rerun_list)} models")

    log_rows   = []
    new_models = {}

    for zone, h in rerun_list:
        df_zone      = df[df["zone"] == zone].copy()
        feature_cols = get_feature_cols(df, df_zone)

        target = f"y_h{h}"
        valid  = df_zone.dropna(subset=feature_cols + [target])

        train = valid[valid["year"].isin(TRAIN_YEARS)]
        calib = valid[valid["year"] == CALIB_YEAR]

        X_train = train[feature_cols].values;  y_train = train[target].values
        X_calib = calib[feature_cols].values;  y_calib = calib[target].values

        prev_best = BEST_MAE[(zone, h)]
        print(f"\n{'='*60}")
        print(f"[{zone}] h={h:02d}  best so far = {prev_best:,.0f} m³")
        print(f"{'='*60}")

        ws = WARMSTART.get((zone, h))
        model, params, mae = tune_lightgbm(
            X_train, y_train, X_calib, y_calib,
            zone=zone, warmstart_params=ws,
        )
        new_models[(zone, h)] = (model, mae)

        delta  = mae - prev_best
        symbol = "✅" if delta < 0 else "❌"
        print(f"  {symbol}  รอบ 3 MAE = {mae:,.2f}  |  delta = {delta:+,.0f} m³")

        log_rows.append({
            "zone": zone, "horizon": h,
            "best_prev":    round(prev_best, 2),
            "calib_mae_r3": round(mae, 2),
            "delta":        round(delta, 2),
            **{f"lgb_{k}": v for k, v in params.items()},
        })

    # ── Merge: เก็บดีที่สุดจากทุก round ──────────────────────────────────
    print(f"\n{'='*60}")
    print("Merging results...")
    replaced = 0
    for (zone, h), (model, mae) in new_models.items():
        if mae < BEST_MAE[(zone, h)]:
            all_models[(zone, h)] = model
            replaced += 1
            print(f"  ✅ ({zone}, h={h:2d}) replaced  "
                  f"{BEST_MAE[(zone,h)]:,.0f} → {mae:,.0f}  "
                  f"(Δ={mae-BEST_MAE[(zone,h)]:+,.0f})")
        else:
            print(f"  ⏸  ({zone}, h={h:2d}) kept prev  "
                  f"({BEST_MAE[(zone,h)]:,.0f} ≤ {mae:,.0f})")

    joblib.dump(all_models, "lightgbm_models.pkl")
    print(f"\nSaved lightgbm_models.pkl ({len(all_models)} models)")
    print(f"Replaced: {replaced} / {len(rerun_list)}")

    # ── Summary table ─────────────────────────────────────────────────────
    log_df = pd.DataFrame(log_rows)
    log_df.to_csv("lightgbm_tuning_log_r3.csv", index=False)

    print(f"\n{'='*55}")
    print("Round 3 summary")
    print("="*55)
    for zone in ["zone_A", "zone_B"]:
        sub = log_df[log_df["zone"] == zone][
            ["zone","horizon","best_prev","calib_mae_r3","delta"]]
        imp = (sub["delta"] < 0).sum()
        print(f"\n{zone}  ({imp}/{len(sub)} improved)")
        print(sub.to_string(index=False))


if __name__ == "__main__":
    main()

Loaded lightgbm_models.pkl (24 models)

Total rerun: 18 models

[zone_A] h=02  best so far = 86,248 m³


  0%|          | 0/120 [00:00<?, ?it/s]

Training until validation scores don't improve for 80 rounds
Early stopping, best iteration is:
[1]	valid_0's l1: 78579.8
Training until validation scores don't improve for 80 rounds
Early stopping, best iteration is:
[1]	valid_0's l1: 84453.1
Training until validation scores don't improve for 80 rounds
Early stopping, best iteration is:
[3]	valid_0's l1: 83874.9
Training until validation scores don't improve for 80 rounds
Early stopping, best iteration is:
[1]	valid_0's l1: 85352.1
Training until validation scores don't improve for 80 rounds
Early stopping, best iteration is:
[4]	valid_0's l1: 78148.9
Training until validation scores don't improve for 80 rounds
Early stopping, best iteration is:
[10]	valid_0's l1: 81979.7
Training until validation scores don't improve for 80 rounds
Early stopping, best iteration is:
[1]	valid_0's l1: 83564
Training until validation scores don't improve for 80 rounds
Early stopping, best iteration is:
[1]	valid_0's l1: 79630.9
Training until validation

  0%|          | 0/120 [00:00<?, ?it/s]

Training until validation scores don't improve for 80 rounds
Early stopping, best iteration is:
[27]	valid_0's l1: 72417.1
Training until validation scores don't improve for 80 rounds
Early stopping, best iteration is:
[2]	valid_0's l1: 80297.1
Training until validation scores don't improve for 80 rounds
Early stopping, best iteration is:
[5]	valid_0's l1: 79787.9
Training until validation scores don't improve for 80 rounds
Early stopping, best iteration is:
[12]	valid_0's l1: 78271.2
Training until validation scores don't improve for 80 rounds
Early stopping, best iteration is:
[9]	valid_0's l1: 78342.5
Training until validation scores don't improve for 80 rounds
Early stopping, best iteration is:
[5]	valid_0's l1: 78277.1
Training until validation scores don't improve for 80 rounds
Early stopping, best iteration is:
[7]	valid_0's l1: 82552.8
Training until validation scores don't improve for 80 rounds
Early stopping, best iteration is:
[5]	valid_0's l1: 81329.8
Training until validat

...

Training until validation scores don't improve for 80 rounds
Early stopping, best iteration is:
[15]	valid_0's l1: 27525
Training until validation scores don't improve for 80 rounds
Early stopping, best iteration is:
[5]	valid_0's l1: 26663.9
Training until validation scores don't improve for 80 rounds
Early stopping, best iteration is:
[68]	valid_0's l1: 28730.7
Training until validation scores don't improve for 80 rounds
Early stopping, best iteration is:
[72]	valid_0's l1: 28021.9
Training until validation scores don't improve for 80 rounds
Early stopping, best iteration is:
[56]	valid_0's l1: 27796.7
Training until validation scores don't improve for 80 rounds
Early stopping, best iteration is:
[100]	valid_0's l1: 28397.8
Training until validation scores don't improve for 80 rounds
Early stopping, best iteration is:
[49]	valid_0's l1: 28063.3
Training until validation scores don't improve for 80 rounds
Early stopping, best iteration is:
[21]	valid_0's l1: 27897.7
Training until val

  0%|          | 0/120 [00:00<?, ?it/s]

Training until validation scores don't improve for 80 rounds
Early stopping, best iteration is:
[15]	valid_0's l1: 27894.6
Training until validation scores don't improve for 80 rounds
Early stopping, best iteration is:
[2]	valid_0's l1: 29396.1
Training until validation scores don't improve for 80 rounds
Early stopping, best iteration is:
[34]	valid_0's l1: 30313
Training until validation scores don't improve for 80 rounds
Early stopping, best iteration is:
[23]	valid_0's l1: 29797.7
Training until validation scores don't improve for 80 rounds
Early stopping, best iteration is:
[9]	valid_0's l1: 30038.2
Training until validation scores don't improve for 80 rounds
Early stopping, best iteration is:
[18]	valid_0's l1: 30571.1
Training until validation scores don't improve for 80 rounds
Early stopping, best iteration is:
[8]	valid_0's l1: 30979.8
Training until validation scores don't improve for 80 rounds
Early stopping, best iteration is:
[4]	valid_0's l1: 30819.9
Training until validat

  0%|          | 0/120 [00:00<?, ?it/s]

Training until validation scores don't improve for 80 rounds
Early stopping, best iteration is:
[33]	valid_0's l1: 28069.2
Training until validation scores don't improve for 80 rounds
Early stopping, best iteration is:
[26]	valid_0's l1: 25767.3
Training until validation scores don't improve for 80 rounds
Early stopping, best iteration is:
[82]	valid_0's l1: 27789.8
Training until validation scores don't improve for 80 rounds
Early stopping, best iteration is:
[66]	valid_0's l1: 27668
Training until validation scores don't improve for 80 rounds
Early stopping, best iteration is:
[38]	valid_0's l1: 27564.4
Training until validation scores don't improve for 80 rounds
Early stopping, best iteration is:
[70]	valid_0's l1: 28131.1
Training until validation scores don't improve for 80 rounds
Early stopping, best iteration is:
[26]	valid_0's l1: 27151.1
Training until validation scores don't improve for 80 rounds
Early stopping, best iteration is:
[5]	valid_0's l1: 29946.8
Training until vali

In [20]:
"""
Fix: Zone B CatBoost + LightGBM → 37 features
================================================
ปัญหา : Zone B models ทั้ง 2 ถูก train ด้วย 38 features (GIR_B_m3 หลุดเข้าไป)
         แต่ step3c ต้องการ 37 features → mismatch

วิธีแก้: retrain ด้วย best params เดิม แต่ใช้ 37-feature set (GIR_B_m3 excluded)
         ไม่ต้อง Optuna ใหม่ — ใช้ params จาก tuning log ที่บันทึกไว้

Input  : ml_features_phase4.csv
         catboost_models.pkl
         lightgbm_models.pkl
Output : catboost_models.pkl  ← Zone B fixed (37 features)
         lightgbm_models.pkl  ← Zone B fixed (37 features, ยกเว้น h=9 ที่ OK อยู่แล้ว)
         fix_feature_log.csv
"""

import pandas as pd
import numpy as np
import joblib
import warnings
warnings.filterwarnings("ignore")

from catboost import CatBoostRegressor
import lightgbm as lgb
from sklearn.metrics import mean_absolute_error

HORIZON     = 12
TRAIN_YEARS = [2020, 2021, 2022]
CALIB_YEAR  = 2023

# ── Best params Zone B ─────────────────────────────────────────────────────
# คัดจาก tuning log — round ที่ให้ calib_mae ต่ำสุดต่อ horizon

CAT_B_PARAMS = {
    1:  {"iterations":1361, "depth":8,  "learning_rate":0.1099, "l2_leaf_reg":1.886,
         "subsample":0.632, "colsample_bylevel":0.676, "min_data_in_leaf":16},
    2:  {"iterations":1311, "depth":10, "learning_rate":0.2788, "l2_leaf_reg":3.136,
         "subsample":0.754, "colsample_bylevel":0.658, "min_data_in_leaf":14},
    3:  {"iterations":1634, "depth":6,  "learning_rate":0.2015, "l2_leaf_reg":2.342,
         "subsample":0.826, "colsample_bylevel":0.765, "min_data_in_leaf":16},
    4:  {"iterations": 881, "depth":6,  "learning_rate":0.1496, "l2_leaf_reg":4.192,
         "subsample":0.914, "colsample_bylevel":0.600, "min_data_in_leaf":11},
    5:  {"iterations":1350, "depth":5,  "learning_rate":0.2496, "l2_leaf_reg":2.047,
         "subsample":0.934, "colsample_bylevel":0.673, "min_data_in_leaf": 5},
    6:  {"iterations":1615, "depth":6,  "learning_rate":0.1047, "l2_leaf_reg":2.910,
         "subsample":0.783, "colsample_bylevel":0.779, "min_data_in_leaf":15},
    7:  {"iterations":1229, "depth":5,  "learning_rate":0.2287, "l2_leaf_reg":7.985,
         "subsample":0.675, "colsample_bylevel":0.816, "min_data_in_leaf":20},
    8:  {"iterations": 907, "depth":5,  "learning_rate":0.2148, "l2_leaf_reg":5.882,
         "subsample":0.971, "colsample_bylevel":0.521, "min_data_in_leaf":19},
    9:  {"iterations":1645, "depth":5,  "learning_rate":0.2090, "l2_leaf_reg":7.545,
         "subsample":0.603, "colsample_bylevel":0.707, "min_data_in_leaf":12},
    10: {"iterations":1388, "depth":5,  "learning_rate":0.1961, "l2_leaf_reg":1.075,
         "subsample":0.719, "colsample_bylevel":0.996, "min_data_in_leaf": 6},
    11: {"iterations":1960, "depth":5,  "learning_rate":0.2885, "l2_leaf_reg":3.642,
         "subsample":0.772, "colsample_bylevel":0.903, "min_data_in_leaf": 5},
    12: {"iterations":1623, "depth":6,  "learning_rate":0.2086, "l2_leaf_reg":1.061,
         "subsample":0.870, "colsample_bylevel":0.724, "min_data_in_leaf":13},
}

# h=9 ข้ามเพราะ LGB Zone B h=9 มี 37 features อยู่แล้ว (R3)
LGB_B_PARAMS = {
    1:  {"n_estimators":1519, "learning_rate":0.1842, "num_leaves": 94, "max_depth": 5,
         "min_child_samples":16, "subsample":0.637, "colsample_bytree":0.499,
         "reg_alpha":2.223,  "reg_lambda":0.000},
    2:  {"n_estimators":1641, "learning_rate":0.1108, "num_leaves":115, "max_depth":11,
         "min_child_samples":12, "subsample":0.618, "colsample_bytree":0.490,
         "reg_alpha":0.002,  "reg_lambda":0.017},
    3:  {"n_estimators": 724, "learning_rate":0.1457, "num_leaves": 94, "max_depth":11,
         "min_child_samples": 8, "subsample":0.582, "colsample_bytree":0.504,
         "reg_alpha":0.009,  "reg_lambda":0.001},
    4:  {"n_estimators":1670, "learning_rate":0.0525, "num_leaves":110, "max_depth":12,
         "min_child_samples":17, "subsample":0.975, "colsample_bytree":0.975,
         "reg_alpha":6.784,  "reg_lambda":0.017},
    5:  {"n_estimators":1030, "learning_rate":0.1644, "num_leaves": 23, "max_depth": 3,
         "min_child_samples":12, "subsample":0.670, "colsample_bytree":0.989,
         "reg_alpha":0.001,  "reg_lambda":2.371},
    6:  {"n_estimators":1229, "learning_rate":0.2612, "num_leaves": 46, "max_depth": 7,
         "min_child_samples": 8, "subsample":0.692, "colsample_bytree":0.423,
         "reg_alpha":0.000,  "reg_lambda":1.161},
    7:  {"n_estimators":1501, "learning_rate":0.2966, "num_leaves": 30, "max_depth": 9,
         "min_child_samples":45, "subsample":0.957, "colsample_bytree":0.735,
         "reg_alpha":0.001,  "reg_lambda":0.008},
    8:  {"n_estimators":1992, "learning_rate":0.1216, "num_leaves": 46, "max_depth":12,
         "min_child_samples": 8, "subsample":0.999, "colsample_bytree":0.843,
         "reg_alpha":0.015,  "reg_lambda":0.167},
    # h=9 skip — already 37 features (R3)
    10: {"n_estimators":1025, "learning_rate":0.2855, "num_leaves":150, "max_depth": 9,
         "min_child_samples": 7, "subsample":0.956, "colsample_bytree":0.545,
         "reg_alpha":0.004,  "reg_lambda":0.005},
    11: {"n_estimators": 872, "learning_rate":0.2981, "num_leaves": 61, "max_depth": 3,
         "min_child_samples": 6, "subsample":0.694, "colsample_bytree":0.821,
         "reg_alpha":0.005,  "reg_lambda":4.910},
    12: {"n_estimators": 724, "learning_rate":0.1610, "num_leaves": 84, "max_depth":10,
         "min_child_samples":20, "subsample":0.737, "colsample_bytree":0.708,
         "reg_alpha":0.007,  "reg_lambda":0.074},
}


def get_feature_cols(df: pd.DataFrame, df_zone: pd.DataFrame = None) -> list:
    """37-feature version — GIR_B_m3 correctly excluded."""
    exclude = {"year", "week", "month", "date", "zone", "target_col",
               "NIR_A_m3", "GIR_B_m3", "P_4week"}
    exclude |= {f"y_h{h}" for h in range(1, HORIZON + 1)}
    cols = [c for c in df.columns if c not in exclude]
    if df_zone is not None:
        cols = [c for c in cols if not df_zone[c].isna().all()]
    return cols


def fix_catboost_zone_b(df, df_zone, feat_cols, all_models):
    print("\n" + "="*55)
    print("Fixing CatBoost Zone B (38 → 37 features)")
    print("="*55)
    log = []
    for h in range(1, HORIZON + 1):
        target = f"y_h{h}"
        valid  = df_zone.dropna(subset=feat_cols + [target])
        train  = valid[valid["year"].isin(TRAIN_YEARS)]
        calib  = valid[valid["year"] == CALIB_YEAR]
        X_tr, y_tr = train[feat_cols].values, train[target].values
        X_ca, y_ca = calib[feat_cols].values, calib[target].values

        params = {**CAT_B_PARAMS[h], "verbose": False, "random_seed": 42}
        model  = CatBoostRegressor(**params)
        model.fit(X_tr, y_tr)
        n = len(model.feature_importances_)
        mae = mean_absolute_error(y_ca, model.predict(X_ca))

        all_models[("zone_B", h)] = model
        log.append({"model":"CatBoost","zone":"zone_B","horizon":h,
                    "n_features":n, "calib_mae":round(mae, 2)})
        print(f"  h={h:02d} | features={n} | MAE={mae:,.2f}")
    return log


def fix_lgb_zone_b(df, df_zone, feat_cols, all_models):
    print("\n" + "="*55)
    print("Fixing LightGBM Zone B (38 → 37 features, skip h=9)")
    print("="*55)
    log = []
    for h in range(1, HORIZON + 1):
        if h == 9:
            n = all_models[("zone_B", 9)].n_features_in_
            print(f"  h=09 | features={n} ✓ skipped (already 37)")
            log.append({"model":"LightGBM","zone":"zone_B","horizon":9,
                        "n_features":n, "calib_mae":"kept"})
            continue

        target = f"y_h{h}"
        valid  = df_zone.dropna(subset=feat_cols + [target])
        train  = valid[valid["year"].isin(TRAIN_YEARS)]
        calib  = valid[valid["year"] == CALIB_YEAR]
        X_tr, y_tr = train[feat_cols].values, train[target].values
        X_ca, y_ca = calib[feat_cols].values, calib[target].values

        params = {
            "objective": "regression", "metric": "mae",
            "verbosity": -1, **LGB_B_PARAMS[h],
        }
        model = lgb.LGBMRegressor(**params)
        model.fit(X_tr, y_tr,
                  eval_set=[(X_ca, y_ca)],
                  callbacks=[lgb.early_stopping(50), lgb.log_evaluation(-1)])
        n   = model.n_features_in_
        mae = mean_absolute_error(y_ca, model.predict(X_ca))

        all_models[("zone_B", h)] = model
        log.append({"model":"LightGBM","zone":"zone_B","horizon":h,
                    "n_features":n, "calib_mae":round(mae, 2)})
        print(f"  h={h:02d} | features={n} | MAE={mae:,.2f}")
    return log


def verify(cat_models, lgb_models):
    print("\n" + "="*55)
    print("Verification — feature counts")
    print("="*55)
    ok = True
    for zone in ["zone_A", "zone_B"]:
        for h in range(1, HORIZON + 1):
            nc = len(cat_models[(zone, h)].feature_importances_)
            nl = lgb_models[(zone, h)].n_features_in_
            status = "✅" if nc == 37 and nl == 37 else "❌"
            if nc != 37 or nl != 37:
                ok = False
            print(f"  {status} ({zone}, h={h:2d}) Cat={nc}  LGB={nl}")
    print()
    print("All 37 ✅" if ok else "⚠️  Still mismatched — check above")


def main():
    df        = pd.read_csv("ml_features_phase4.csv")
    cat_all   = joblib.load("catboost_models.pkl")
    lgb_all   = joblib.load("lightgbm_models.pkl")

    zone    = "zone_B"
    df_zone = df[df["zone"] == zone].copy()
    feat_cols = get_feature_cols(df, df_zone)
    print(f"Feature cols: {len(feat_cols)}  (expect 37)")

    log  = fix_catboost_zone_b(df, df_zone, feat_cols, cat_all)
    log += fix_lgb_zone_b(df, df_zone, feat_cols, lgb_all)

    # Save
    joblib.dump(cat_all, "catboost_models.pkl")
    joblib.dump(lgb_all, "lightgbm_models.pkl")
    print("\n✅ Saved catboost_models.pkl")
    print("✅ Saved lightgbm_models.pkl")

    pd.DataFrame(log).to_csv("fix_feature_log.csv", index=False)
    print("✅ Saved fix_feature_log.csv")

    verify(cat_all, lgb_all)


if __name__ == "__main__":
    main()

Feature cols: 37  (expect 37)

Fixing CatBoost Zone B (38 → 37 features)
  h=01 | features=37 | MAE=27,243.19
  h=02 | features=37 | MAE=29,828.71
  h=03 | features=37 | MAE=28,227.07
  h=04 | features=37 | MAE=28,093.30
  h=05 | features=37 | MAE=28,611.82
  h=06 | features=37 | MAE=25,726.50
  h=07 | features=37 | MAE=29,825.17
  h=08 | features=37 | MAE=28,405.16
  h=09 | features=37 | MAE=27,874.63
  h=10 | features=37 | MAE=29,096.41
  h=11 | features=37 | MAE=30,922.40
  h=12 | features=37 | MAE=28,043.13

Fixing LightGBM Zone B (38 → 37 features, skip h=9)
Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[30]	valid_0's l1: 27874
  h=01 | features=37 | MAE=27,874.01
Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[1]	valid_0's l1: 30185.7
  h=02 | features=37 | MAE=30,185.70
Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[21]	valid_0's l1

In [21]:
"""
Step 3c — Ridge Meta-Learner (Stacking)
========================================
Input  : ml_features_phase4.csv
         catboost_models.pkl     ← จาก step3a
         lightgbm_models.pkl     ← จาก step3b
Output : ridge_meta_models.pkl
         stacking_summary.csv    (calib + test MAE per zone × horizon)

Run AFTER step3a และ step3b เสร็จแล้ว:
    python step3c_ridge_meta.py
"""

import pandas as pd
import numpy as np
import joblib

from sklearn.linear_model import RidgeCV
from sklearn.metrics import mean_absolute_error

# ── Config ────────────────────────────────────────────────────────────────────
HORIZON     = 12
ZONES       = ["zone_A", "zone_B"]
TRAIN_YEARS = [2020, 2021, 2022]
CALIB_YEAR  = 2023
TEST_YEAR   = 2024

# RidgeCV — ค้นหา alpha อัตโนมัติจาก list นี้
RIDGE_ALPHAS = [0.01, 0.1, 0.5, 1.0, 5.0, 10.0, 50.0, 100.0]


def get_feature_cols(df: pd.DataFrame, df_zone: pd.DataFrame = None) -> list:
    exclude = {"year", "week", "month", "date", "zone", "target_col",
               "NIR_A_m3", "GIR_B_m3", "P_4week"}
    exclude |= {f"y_h{h}" for h in range(1, HORIZON + 1)}
    cols = [c for c in df.columns if c not in exclude]
    if df_zone is not None:
        cols = [c for c in cols if not df_zone[c].isna().all()]
    return cols


def get_meta_features(cat_models, lgb_models, zone, h, X):
    """Stack predictions from CatBoost + LightGBM as meta-features."""
    cat_pred = cat_models[(zone, h)].predict(X)
    lgb_pred = lgb_models[(zone, h)].predict(X)
    return np.column_stack([cat_pred, lgb_pred])


def main():
    df         = pd.read_csv("ml_features_phase4.csv")
    cat_models = joblib.load("catboost_models.pkl")
    lgb_models = joblib.load("lightgbm_models.pkl")


    meta_models = {}   # key: (zone, h)  value: fitted RidgeCV
    log_rows    = []

    for zone in ZONES:
        df_zone = df[df["zone"] == zone].copy()
        feature_cols = get_feature_cols(df, df_zone)
        print(f"\n{'='*60}")
        print(f"Zone: {zone}")
        print(f"{'='*60}")

        for h in range(1, HORIZON + 1):
            target = f"y_h{h}"
            valid  = df_zone.dropna(subset=feature_cols + [target])

            train = valid[valid["year"].isin(TRAIN_YEARS)]
            calib = valid[valid["year"] == CALIB_YEAR]
            test  = valid[valid["year"] == TEST_YEAR]

            X_train = train[feature_cols].values
            y_train = train[target].values
            X_calib = calib[feature_cols].values
            y_calib = calib[target].values
            X_test  = test[feature_cols].values
            y_test  = test[target].values

            # ── Build meta-feature matrices ───────────────────────────────
            # Train meta on train + calib combined
            # (base models were fit on train only, so calib is out-of-fold)
            meta_train = get_meta_features(cat_models, lgb_models, zone, h, X_train)
            meta_calib = get_meta_features(cat_models, lgb_models, zone, h, X_calib)
            meta_test  = get_meta_features(cat_models, lgb_models, zone, h, X_test)

            meta_X_fit = np.vstack([meta_train, meta_calib])
            meta_y_fit = np.concatenate([y_train, y_calib])

            # ── RidgeCV (finds best alpha via leave-one-out CV) ───────────
            ridge = RidgeCV(alphas=RIDGE_ALPHAS, cv=5)
            ridge.fit(meta_X_fit, meta_y_fit)
            meta_models[(zone, h)] = ridge

            # ── Evaluate ─────────────────────────────────────────────────
            pred_calib_stack = ridge.predict(meta_calib)
            pred_test_stack  = ridge.predict(meta_test)
            mae_calib = mean_absolute_error(y_calib, pred_calib_stack)
            mae_test  = mean_absolute_error(y_test,  pred_test_stack)

            # Individual model MAE on test (for comparison)
            mae_cat  = mean_absolute_error(y_test, cat_models[(zone, h)].predict(X_test))
            mae_lgb  = mean_absolute_error(y_test, lgb_models[(zone, h)].predict(X_test))

            log_rows.append({
                "zone":         zone,
                "horizon":      h,
                "ridge_alpha":  ridge.alpha_,
                "mae_calib_stack": round(mae_calib, 2),
                "mae_test_cat":    round(mae_cat, 2),
                "mae_test_lgb":    round(mae_lgb, 2),
                "mae_test_stack":  round(mae_test, 2),
                "ridge_coef_cat":  round(ridge.coef_[0], 4),
                "ridge_coef_lgb":  round(ridge.coef_[1], 4),
            })

            print(f"  h={h:02d} | alpha={ridge.alpha_:.2f} "
                  f"| MAE calib={mae_calib:.1f} "
                  f"| MAE test → CatBoost={mae_cat:.1f} "
                  f"LGB={mae_lgb:.1f} "
                  f"Stack={mae_test:.1f} m³")

    # ── Save ──────────────────────────────────────────────────────────────
    joblib.dump(meta_models, "ridge_meta_models.pkl")
    summary = pd.DataFrame(log_rows)
    summary.to_csv("stacking_summary.csv", index=False)

    print(f"\nSaved ridge_meta_models.pkl  ({len(meta_models)} models)")
    print("Saved stacking_summary.csv\n")

    # ── Print pivot table: Stack MAE by zone × horizon ────────────────────
    for zone in ZONES:
        sub = summary[summary["zone"] == zone]
        print(f"\nTest MAE comparison — {zone}  (m³/week)")
        print(sub[["horizon","mae_test_cat","mae_test_lgb","mae_test_stack"]]
              .rename(columns={"mae_test_cat":"CatBoost",
                               "mae_test_lgb":"LightGBM",
                               "mae_test_stack":"Stack"})
              .to_string(index=False))


if __name__ == "__main__":
    main()


Zone: zone_A
  h=01 | alpha=0.01 | MAE calib=79466.5 | MAE test → CatBoost=78383.7 LGB=70061.1 Stack=81845.8 m³
  h=02 | alpha=100.00 | MAE calib=86331.3 | MAE test → CatBoost=78464.1 LGB=81377.2 Stack=83533.9 m³
  h=03 | alpha=0.01 | MAE calib=78152.6 | MAE test → CatBoost=78659.9 LGB=71218.4 Stack=80117.0 m³
  h=04 | alpha=0.01 | MAE calib=82754.6 | MAE test → CatBoost=66076.8 LGB=67476.4 Stack=69460.7 m³
  h=05 | alpha=0.01 | MAE calib=74714.2 | MAE test → CatBoost=69628.2 LGB=73465.5 Stack=70338.4 m³
  h=06 | alpha=100.00 | MAE calib=73264.8 | MAE test → CatBoost=78653.5 LGB=71621.2 Stack=82792.1 m³
  h=07 | alpha=0.01 | MAE calib=83245.0 | MAE test → CatBoost=89932.9 LGB=79798.2 Stack=92228.4 m³
  h=08 | alpha=100.00 | MAE calib=82915.3 | MAE test → CatBoost=76859.1 LGB=80701.6 Stack=79218.6 m³
  h=09 | alpha=100.00 | MAE calib=77098.6 | MAE test → CatBoost=81699.5 LGB=81185.3 Stack=82211.3 m³
  h=10 | alpha=100.00 | MAE calib=83502.5 | MAE test → CatBoost=83094.5 LGB=85085.4 Sta

In [1]:
"""
Step 3c v2 — Inverse-MAE Weighted Averaging
=============================================
แทนที่ Ridge Stacking ด้วย Inverse-MAE Weighted Average

เหตุผล: Ridge Stacking ล้มเหลวเพราะ meta-learner เทรนบน in-sample
predictions ของ base models → data leakage → negative weights →
test MAE แย่กว่า base models ทั้ง 2 (Stack wins แค่ 1/24)

วิธีใหม่:
  w_cat  = (1 / mae_calib_cat) / (1/mae_calib_cat + 1/mae_calib_lgb)
  w_lgb  = 1 - w_cat
  ŷ_stack = w_cat * ŷ_cat + w_lgb * ŷ_lgb

ข้อดี:
  - ไม่มี data leakage (ใช้แค่ calib set ซึ่งเป็น out-of-training)
  - interpretable ทุก weight เป็นบวก และรวมกันได้ 1
  - robust กับ dataset ขนาดเล็ก (52 calib weeks)

Input  : ml_features_phase4.csv
         catboost_models.pkl
         lightgbm_models.pkl
Output : stack_weights.pkl        ← (w_cat, w_lgb) per (zone, h)
         stacking_summary_v2.csv
"""

import pandas as pd
import numpy as np
import joblib
from sklearn.metrics import mean_absolute_error

HORIZON     = 12
ZONES       = ["zone_A", "zone_B"]
TRAIN_YEARS = [2020, 2021, 2022]
CALIB_YEAR  = 2023
TEST_YEAR   = 2024


def get_feature_cols(df: pd.DataFrame, df_zone: pd.DataFrame = None) -> list:
    exclude = {"year", "week", "month", "date", "zone", "target_col",
               "NIR_A_m3", "GIR_B_m3", "P_4week"}
    exclude |= {f"y_h{h}" for h in range(1, HORIZON + 1)}
    cols = [c for c in df.columns if c not in exclude]
    if df_zone is not None:
        cols = [c for c in cols if not df_zone[c].isna().all()]
    return cols


def inverse_mae_weights(mae_cat: float, mae_lgb: float) -> tuple:
    """คำนวณน้ำหนัก inverse-MAE ที่รวมกันได้ 1."""
    w_cat = (1 / mae_cat) / (1 / mae_cat + 1 / mae_lgb)
    w_lgb = 1 - w_cat
    return round(w_cat, 4), round(w_lgb, 4)


def nse(y_obs, y_pred):
    return 1 - np.sum((y_obs - y_pred)**2) / np.sum((y_obs - np.mean(y_obs))**2)


def kge(y_obs, y_pred):
    r     = np.corrcoef(y_obs, y_pred)[0, 1]
    alpha = np.std(y_pred) / np.std(y_obs)
    beta  = np.mean(y_pred) / np.mean(y_obs)
    return 1 - np.sqrt((r - 1)**2 + (alpha - 1)**2 + (beta - 1)**2)


def main():
    df         = pd.read_csv("ml_features_phase4.csv")
    cat_models = joblib.load("catboost_models.pkl")
    lgb_models = joblib.load("lightgbm_models.pkl")

    # verify feature counts สม่ำเสมอ
    feat_issues = []
    for zone in ZONES:
        df_zone = df[df["zone"] == zone].copy()
        fc = get_feature_cols(df, df_zone)
        for h in range(1, HORIZON + 1):
            nc = len(cat_models[(zone, h)].feature_importances_)
            nl = lgb_models[(zone, h)].n_features_in_
            exp = len(fc)
            if nc != exp or nl != exp:
                feat_issues.append(f"({zone},h={h}): cat={nc} lgb={nl} expected={exp}")
    if feat_issues:
        print("⚠️  Feature mismatch detected — run fix_zone_b_features.py first:")
        for x in feat_issues:
            print(f"   {x}")
        return
    print("✅ Feature counts verified (all 37)")

    stack_weights = {}   # key: (zone, h) → {"w_cat": .., "w_lgb": ..}
    log_rows      = []

    for zone in ZONES:
        df_zone      = df[df["zone"] == zone].copy()
        feature_cols = get_feature_cols(df, df_zone)

        print(f"\n{'='*60}")
        print(f"Zone: {zone}")
        print(f"{'='*60}")

        for h in range(1, HORIZON + 1):
            target = f"y_h{h}"
            valid  = df_zone.dropna(subset=feature_cols + [target])

            calib = valid[valid["year"] == CALIB_YEAR]
            test  = valid[valid["year"] == TEST_YEAR]

            X_calib = calib[feature_cols].values;  y_calib = calib[target].values
            X_test  = test[feature_cols].values;   y_test  = test[target].values

            # ── Calib predictions ─────────────────────────────────────────
            cat_pred_calib = cat_models[(zone, h)].predict(X_calib)
            lgb_pred_calib = lgb_models[(zone, h)].predict(X_calib)

            mae_cat_calib = mean_absolute_error(y_calib, cat_pred_calib)
            mae_lgb_calib = mean_absolute_error(y_calib, lgb_pred_calib)

            # ── Compute weights ───────────────────────────────────────────
            w_cat, w_lgb = inverse_mae_weights(mae_cat_calib, mae_lgb_calib)
            stack_weights[(zone, h)] = {"w_cat": w_cat, "w_lgb": w_lgb}

            # ── Test predictions ──────────────────────────────────────────
            cat_pred_test = cat_models[(zone, h)].predict(X_test)
            lgb_pred_test = lgb_models[(zone, h)].predict(X_test)
            stk_pred_test = w_cat * cat_pred_test + w_lgb * lgb_pred_test
            stk_pred_test = np.maximum(stk_pred_test, 0)   # demand ≥ 0

            mae_cat  = mean_absolute_error(y_test, cat_pred_test)
            mae_lgb  = mean_absolute_error(y_test, lgb_pred_test)
            mae_stk  = mean_absolute_error(y_test, stk_pred_test)
            nse_stk  = nse(y_test, stk_pred_test)
            kge_stk  = kge(y_test, stk_pred_test)

            winner = "Cat" if mae_cat <= min(mae_lgb, mae_stk) else \
                     "LGB" if mae_lgb <= mae_stk else "Stack"

            print(f"  h={h:02d} | w=[{w_cat:.3f},{w_lgb:.3f}] "
                  f"| test MAE: Cat={mae_cat:.0f}  LGB={mae_lgb:.0f}  "
                  f"Stack={mae_stk:.0f}  [best: {winner}]")

            log_rows.append({
                "zone": zone, "horizon": h,
                "w_cat": w_cat, "w_lgb": w_lgb,
                "mae_calib_cat": round(mae_cat_calib, 2),
                "mae_calib_lgb": round(mae_lgb_calib, 2),
                "mae_test_cat":  round(mae_cat,  2),
                "mae_test_lgb":  round(mae_lgb,  2),
                "mae_test_stack":round(mae_stk,  2),
                "nse_test_stack":round(nse_stk,  4),
                "kge_test_stack":round(kge_stk,  4),
                "best_model": winner,
            })

    # ── Save ──────────────────────────────────────────────────────────────
    joblib.dump(stack_weights, "stack_weights.pkl")
    summary = pd.DataFrame(log_rows)
    summary.to_csv("stacking_summary_v2.csv", index=False)

    print(f"\nSaved stack_weights.pkl  ({len(stack_weights)} entries)")
    print("Saved stacking_summary_v2.csv\n")

    # ── Summary ───────────────────────────────────────────────────────────
    for zone in ZONES:
        sub = summary[summary["zone"] == zone]
        avg_cat = sub.mae_test_cat.mean()
        avg_lgb = sub.mae_test_lgb.mean()
        avg_stk = sub.mae_test_stack.mean()
        stk_wins = (sub.mae_test_stack == sub[["mae_test_cat",
                    "mae_test_lgb","mae_test_stack"]].min(axis=1)).sum()
        print(f"{zone}:")
        print(f"  avg test MAE — Cat={avg_cat:,.0f}  LGB={avg_lgb:,.0f}  "
              f"Stack={avg_stk:,.0f}")
        print(f"  Stack wins: {stk_wins}/12  "
              f"beats Cat: {(sub.mae_test_stack<sub.mae_test_cat).sum()}/12  "
              f"beats LGB: {(sub.mae_test_stack<sub.mae_test_lgb).sum()}/12")
        print(f"  avg NSE={sub.nse_test_stack.mean():.4f}  "
              f"avg KGE={sub.kge_test_stack.mean():.4f}")
        print()

    print("─"*55)
    print("Best model count across all 24 models:")
    print(summary.best_model.value_counts().to_string())


if __name__ == "__main__":
    main()

✅ Feature counts verified (all 37)

Zone: zone_A


C:\Users\mpdox\AppData\Roaming\Python\Python314\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


  h=01 | w=[0.503,0.496] | test MAE: Cat=78384  LGB=70061  Stack=73653  [best: LGB]
  h=02 | w=[0.491,0.509] | test MAE: Cat=78464  LGB=81377  Stack=79770  [best: Cat]
  h=03 | w=[0.505,0.495] | test MAE: Cat=78660  LGB=71218  Stack=74365  [best: LGB]
  h=04 | w=[0.493,0.507] | test MAE: Cat=66077  LGB=67476  Stack=66506  [best: Cat]
  h=05 | w=[0.522,0.478] | test MAE: Cat=69628  LGB=73465  Stack=70440  [best: Cat]
  h=06 | w=[0.511,0.489] | test MAE: Cat=78653  LGB=71621  Stack=74547  [best: LGB]
  h=07 | w=[0.492,0.508] | test MAE: Cat=89933  LGB=79798  Stack=84789  [best: LGB]
  h=08 | w=[0.489,0.511] | test MAE: Cat=76859  LGB=80702  Stack=78771  [best: Cat]
  h=09 | w=[0.483,0.517] | test MAE: Cat=81699  LGB=81185  Stack=80782  [best: Stack]
  h=10 | w=[0.493,0.507] | test MAE: Cat=83094  LGB=85085  Stack=82245  [best: Stack]
  h=11 | w=[0.495,0.505] | test MAE: Cat=75444  LGB=80627  Stack=76928  [best: Cat]
  h=12 | w=[0.487,0.513] | test MAE: Cat=89245  LGB=93326  Stack=90672  

C:\Users\mpdox\AppData\Roaming\Python\Python314\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
C:\Users\mpdox\AppData\Roaming\Python\Python314\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
C:\Users\mpdox\AppData\Roaming\Python\Python314\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
C:\Users\mpdox\AppData\Roaming\Python\Python314\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
C:\Users\mpdox\AppData\Roaming\Python\Python314\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMRe

In [1]:
"""
Step 3d — Stage 1: Binary Classifier (Two-stage Framework)
===========================================================
ทำนายว่าสัปดาห์ h สัปดาห์ข้างหน้าจะมี demand > 0 หรือไม่

Two-stage final prediction:
  ŷ_final = P(demand > 0) × ŷ_magnitude
  → ถ้า P ต่ำ: prediction ถูก pull toward 0 อัตโนมัติ
  → ถ้า P สูง: ใช้ magnitude จาก CatBoost+LGB stack เดิม

Input  : ml_features_phase4.csv
Output : stage1_classifiers.pkl    ← LGBMClassifier per (zone, h)
         stage1_thresholds.pkl     ← optimal threshold per (zone, h)
         stage1_report.csv         ← F1, precision, recall, AUC per model

Design:
  - LightGBM classifier + Optuna tuning
  - class_weight='balanced' (training zeros 37-44%)
  - Threshold tuning บน calib 2023 (maximize F1)
  - Evaluate บน test 2024: precision, recall, F1, AUC

Run: python step3d_stage1_classifier.py
"""

import pandas as pd
import numpy as np
import optuna
import joblib
import warnings
warnings.filterwarnings("ignore")
optuna.logging.set_verbosity(optuna.logging.WARNING)

import lightgbm as lgb
from sklearn.metrics import (f1_score, precision_score, recall_score,
                             roc_auc_score, classification_report,
                             confusion_matrix)

# ── Config ────────────────────────────────────────────────────────────────────
HORIZON     = 12
ZONES       = ["zone_A", "zone_B"]
TRAIN_YEARS = [2020, 2021, 2022]
CALIB_YEAR  = 2023
TEST_YEAR   = 2024
N_TRIALS    = 80
EARLY_STOP  = 60


# ── Classifier feature set (subset ที่เหมาะกับ regime detection) ─────────────
# ใช้ seasonality + recent demand pattern + drought indicators
# ไม่ใช้ rolling std หรือ climate lag ที่ noisy เกินไป
CLASSIFIER_FEATURES = [
    # Seasonality — สำคัญที่สุดสำหรับ regime detection
    "WoY_sin", "WoY_cos", "MoY_sin", "MoY_cos",
    # Recent demand (จะเป็น 0 ถ้าอยู่ในช่วงแล้ง)
    # ชื่อจะถูก resolve ตาม zone (NIR_A หรือ GIR_B)
    # → ดูใน get_clf_features()
    # Climate drivers
    "ET0_mm_week", "P_mm_week", "P_eff_mm",
    "SPI_4", "drought_flag",
    # ENSO
    "MEI",
    # AI
    "AI_week",
]

# lag columns ของ target (zone-specific) — เพิ่มใน get_clf_features()
TARGET_LAGS = [1, 2, 3, 4]     # สัปดาห์ล่าสุด 4 สัปดาห์


def get_clf_features(df_zone: pd.DataFrame, target_col: str) -> list:
    """
    คืน feature list สำหรับ classifier
    รวม CLASSIFIER_FEATURES + lag columns ของ target zone นั้น
    กรอง all-NaN columns ออก
    """
    lag_cols = [f"{target_col}_lag{k}" for k in TARGET_LAGS]
    roll_cols = [f"{target_col}_roll4_mean",
                 f"{target_col}_roll8_mean"]

    wanted = CLASSIFIER_FEATURES + lag_cols + roll_cols

    # กรองเฉพาะที่มีอยู่จริงและไม่ all-NaN
    cols = [c for c in wanted
            if c in df_zone.columns and not df_zone[c].isna().all()]
    return cols


def make_binary_target(series: pd.Series) -> pd.Series:
    """demand > 0 → 1 (active), demand = 0 → 0 (inactive)"""
    return (series > 0).astype(int)


def tune_classifier(X_train, y_train, X_calib, y_calib,
                    class_ratio: float) -> tuple:
    """
    Optuna tuning สำหรับ LGBMClassifier
    class_ratio = n_negative / n_positive (สำหรับ scale_pos_weight)
    Returns (best_model, best_params, best_threshold, best_f1_calib)
    """
    def objective(trial):
        params = {
            "objective":         "binary",
            "metric":            "binary_logloss",
            "verbosity":         -1,
            "n_estimators":      trial.suggest_int("n_estimators", 200, 1500),
            "learning_rate":     trial.suggest_float("learning_rate", 0.01, 0.3, log=True),
            "num_leaves":        trial.suggest_int("num_leaves", 15, 80),
            "max_depth":         trial.suggest_int("max_depth", 3, 10),
            "min_child_samples": trial.suggest_int("min_child_samples", 5, 30),
            "subsample":         trial.suggest_float("subsample", 0.5, 1.0),
            "colsample_bytree":  trial.suggest_float("colsample_bytree", 0.5, 1.0),
            "reg_alpha":         trial.suggest_float("reg_alpha", 1e-4, 5.0, log=True),
            "reg_lambda":        trial.suggest_float("reg_lambda", 1e-4, 5.0, log=True),
            "scale_pos_weight":  class_ratio,   # handle class imbalance
        }
        m = lgb.LGBMClassifier(**params)
        m.fit(X_train, y_train,
              eval_set=[(X_calib, y_calib)],
              callbacks=[lgb.early_stopping(EARLY_STOP), lgb.log_evaluation(-1)])
        # Optimize F1 on calib (better than logloss for imbalanced data)
        prob = m.predict_proba(X_calib)[:, 1]
        # Find best threshold
        thresholds = np.arange(0.2, 0.8, 0.05)
        best_f1 = max(f1_score(y_calib, (prob >= t).astype(int), zero_division=0)
                      for t in thresholds)
        return -best_f1   # minimize negative F1

    study = optuna.create_study(
        direction="minimize",
        sampler=optuna.samplers.TPESampler(seed=42),
    )
    study.optimize(objective, n_trials=N_TRIALS, show_progress_bar=True)

    # Train best model
    best_params = {
        "objective": "binary", "metric": "binary_logloss",
        "verbosity": -1, "scale_pos_weight": class_ratio,
        **study.best_params,
    }
    best_model = lgb.LGBMClassifier(**best_params)
    best_model.fit(X_train, y_train,
                   eval_set=[(X_calib, y_calib)],
                   callbacks=[lgb.early_stopping(EARLY_STOP), lgb.log_evaluation(-1)])

    # Tune threshold on calib
    prob_calib = best_model.predict_proba(X_calib)[:, 1]
    thresholds  = np.arange(0.15, 0.85, 0.025)
    best_thr, best_f1 = 0.5, 0.0
    for t in thresholds:
        f1 = f1_score(y_calib, (prob_calib >= t).astype(int), zero_division=0)
        if f1 > best_f1:
            best_f1, best_thr = f1, t

    return best_model, study.best_params, round(best_thr, 3), round(best_f1, 4)


def main():
    df = pd.read_csv("ml_features_phase4.csv")

    classifiers = {}    # key: (zone, h) → LGBMClassifier
    thresholds  = {}    # key: (zone, h) → float
    log_rows    = []

    for zone in ZONES:
        target_col = "NIR_A_m3" if zone == "zone_A" else "GIR_B_m3"
        df_zone    = df[df["zone"] == zone].copy()
        clf_feats  = get_clf_features(df_zone, target_col)

        print(f"\n{'='*60}")
        print(f"Zone: {zone}  |  target: {target_col}")
        print(f"Classifier features ({len(clf_feats)}): {clf_feats}")
        print(f"{'='*60}")

        # Class distribution per year
        for yr in TRAIN_YEARS + [CALIB_YEAR, TEST_YEAR]:
            yr_data = df_zone[df_zone["year"] == yr][target_col].dropna()
            zeros   = (yr_data == 0).sum()
            print(f"  {yr}: {zeros}/{len(yr_data)} zeros "
                  f"({zeros/len(yr_data)*100:.0f}%)")
        print()

        for h in range(1, HORIZON + 1):
            target_h = f"y_h{h}"

            valid = df_zone.dropna(subset=clf_feats + [target_h])
            train = valid[valid["year"].isin(TRAIN_YEARS)]
            calib = valid[valid["year"] == CALIB_YEAR]
            test  = valid[valid["year"] == TEST_YEAR]

            # Binary labels
            y_tr = make_binary_target(train[target_h])
            y_ca = make_binary_target(calib[target_h])
            y_te = make_binary_target(test[target_h])

            X_tr  = train[clf_feats].values
            X_ca  = calib[clf_feats].values
            X_te  = test[clf_feats].values

            # Class ratio for scale_pos_weight
            n_neg  = (y_tr == 0).sum()
            n_pos  = (y_tr == 1).sum()
            ratio  = n_neg / max(n_pos, 1)

            print(f"  h={h:02d} | train: {n_pos}pos/{n_neg}neg  "
                  f"calib: {y_ca.sum()}/{len(y_ca)}pos  "
                  f"test: {y_te.sum()}/{len(y_te)}pos")

            model, params, thr, f1_calib = tune_classifier(
                X_tr, y_tr, X_ca, y_ca, class_ratio=ratio
            )
            classifiers[(zone, h)] = model
            thresholds[(zone, h)]  = thr

            # Evaluate on TEST set
            prob_te  = model.predict_proba(X_te)[:, 1]
            pred_te  = (prob_te >= thr).astype(int)
            f1_te    = f1_score(y_te, pred_te, zero_division=0)
            prec_te  = precision_score(y_te, pred_te, zero_division=0)
            rec_te   = recall_score(y_te, pred_te, zero_division=0)
            auc_te   = roc_auc_score(y_te, prob_te) if y_te.nunique() > 1 else np.nan
            # Zero-hit: correctly predict inactive (y=0 → pred=0)
            zero_hit = (pred_te[y_te == 0] == 0).mean() * 100 if (y_te==0).sum()>0 else np.nan

            print(f"       threshold={thr:.3f} | calib F1={f1_calib:.3f} "
                  f"| test F1={f1_te:.3f} prec={prec_te:.3f} rec={rec_te:.3f} "
                  f"AUC={auc_te:.3f} zero-hit={zero_hit:.0f}%")

            cm = confusion_matrix(y_te, pred_te)
            print(f"       confusion (test): TN={cm[0,0]} FP={cm[0,1]} "
                  f"FN={cm[1,0]} TP={cm[1,1]}")

            log_rows.append({
                "zone": zone, "horizon": h,
                "threshold":   thr,
                "f1_calib":    round(f1_calib, 4),
                "f1_test":     round(f1_te,    4),
                "precision":   round(prec_te,  4),
                "recall":      round(rec_te,   4),
                "auc":         round(auc_te,   4) if not np.isnan(auc_te) else np.nan,
                "zero_hit_pct": round(zero_hit, 1) if not np.isnan(zero_hit) else np.nan,
                "n_train_pos": int(n_pos), "n_train_neg": int(n_neg),
            })

    # ── Save ──────────────────────────────────────────────────────────────
    joblib.dump(classifiers, "stage1_classifiers.pkl")
    joblib.dump(thresholds,  "stage1_thresholds.pkl")

    report = pd.DataFrame(log_rows)
    report.to_csv("stage1_report.csv", index=False)

    print(f"\n{'='*60}")
    print(f"Saved stage1_classifiers.pkl ({len(classifiers)} models)")
    print(f"Saved stage1_thresholds.pkl")
    print(f"Saved stage1_report.csv")
    print(f"{'='*60}")

    # ── Summary ───────────────────────────────────────────────────────────
    for zone in ZONES:
        sub = report[report["zone"] == zone]
        print(f"\n{zone}:")
        print(f"  avg F1 (test)   = {sub.f1_test.mean():.3f}")
        print(f"  avg precision   = {sub.precision.mean():.3f}")
        print(f"  avg recall      = {sub.recall.mean():.3f}")
        print(f"  avg AUC         = {sub.auc.mean():.3f}")
        print(f"  avg zero-hit    = {sub.zero_hit_pct.mean():.1f}%")
        print(f"  avg threshold   = {sub.threshold.mean():.3f}")
        print()
        print(sub[["horizon","threshold","f1_test","precision",
                   "recall","auc","zero_hit_pct"]].to_string(index=False))


if __name__ == "__main__":
    main()


Zone: zone_A  |  target: NIR_A_m3
Classifier features (17): ['WoY_sin', 'WoY_cos', 'MoY_sin', 'MoY_cos', 'ET0_mm_week', 'P_mm_week', 'P_eff_mm', 'SPI_4', 'drought_flag', 'MEI', 'AI_week', 'NIR_A_m3_lag1', 'NIR_A_m3_lag2', 'NIR_A_m3_lag3', 'NIR_A_m3_lag4', 'NIR_A_m3_roll4_mean', 'NIR_A_m3_roll8_mean']
  2020: 19/52 zeros (37%)
  2021: 18/52 zeros (35%)
  2022: 20/52 zeros (38%)
  2023: 11/52 zeros (21%)
  2024: 15/51 zeros (29%)

  h=01 | train: 91pos/57neg  calib: 41/52pos  test: 36/51pos


  0%|          | 0/80 [00:00<?, ?it/s]

Training until validation scores don't improve for 60 rounds
Early stopping, best iteration is:
[1]	valid_0's binary_logloss: 0.593973
Training until validation scores don't improve for 60 rounds
Early stopping, best iteration is:
[43]	valid_0's binary_logloss: 0.561898
Training until validation scores don't improve for 60 rounds
Early stopping, best iteration is:
[13]	valid_0's binary_logloss: 0.563155
Training until validation scores don't improve for 60 rounds
Early stopping, best iteration is:
[7]	valid_0's binary_logloss: 0.572285
Training until validation scores don't improve for 60 rounds
Early stopping, best iteration is:
[43]	valid_0's binary_logloss: 0.566825
Training until validation scores don't improve for 60 rounds
Early stopping, best iteration is:
[13]	valid_0's binary_logloss: 0.573479
Training until validation scores don't improve for 60 rounds
Early stopping, best iteration is:
[2]	valid_0's binary_logloss: 0.577943
Training until validation scores don't improve for 

  0%|          | 0/80 [00:00<?, ?it/s]

Training until validation scores don't improve for 60 rounds
Early stopping, best iteration is:
[3]	valid_0's binary_logloss: 0.558329
Training until validation scores don't improve for 60 rounds
Early stopping, best iteration is:
[56]	valid_0's binary_logloss: 0.554899
Training until validation scores don't improve for 60 rounds
Early stopping, best iteration is:
[20]	valid_0's binary_logloss: 0.560219
Training until validation scores don't improve for 60 rounds
Early stopping, best iteration is:
[6]	valid_0's binary_logloss: 0.567176
Training until validation scores don't improve for 60 rounds
Early stopping, best iteration is:
[65]	valid_0's binary_logloss: 0.556487
Training until validation scores don't improve for 60 rounds
Early stopping, best iteration is:
[17]	valid_0's binary_logloss: 0.570519
Training until validation scores don't improve for 60 rounds
Early stopping, best iteration is:
[1]	valid_0's binary_logloss: 0.567636
Training until validation scores don't improve for 

...

Training until validation scores don't improve for 60 rounds
Early stopping, best iteration is:
[18]	valid_0's binary_logloss: 0.626775
Training until validation scores don't improve for 60 rounds
Early stopping, best iteration is:
[68]	valid_0's binary_logloss: 0.58312
Training until validation scores don't improve for 60 rounds
Early stopping, best iteration is:
[32]	valid_0's binary_logloss: 0.564061
Training until validation scores don't improve for 60 rounds
Early stopping, best iteration is:
[18]	valid_0's binary_logloss: 0.589868
Training until validation scores don't improve for 60 rounds
Early stopping, best iteration is:
[139]	valid_0's binary_logloss: 0.585491
Training until validation scores don't improve for 60 rounds
Early stopping, best iteration is:
[47]	valid_0's binary_logloss: 0.585068
Training until validation scores don't improve for 60 rounds
Early stopping, best iteration is:
[4]	valid_0's binary_logloss: 0.605359
Training until validation scores don't improve fo

  0%|          | 0/80 [00:00<?, ?it/s]

Training until validation scores don't improve for 60 rounds
Early stopping, best iteration is:
[13]	valid_0's binary_logloss: 0.580637
Training until validation scores don't improve for 60 rounds
Early stopping, best iteration is:
[85]	valid_0's binary_logloss: 0.57407
Training until validation scores don't improve for 60 rounds
Early stopping, best iteration is:
[34]	valid_0's binary_logloss: 0.560546
Training until validation scores don't improve for 60 rounds
Early stopping, best iteration is:
[42]	valid_0's binary_logloss: 0.57388
Training until validation scores don't improve for 60 rounds
Early stopping, best iteration is:
[111]	valid_0's binary_logloss: 0.573916
Training until validation scores don't improve for 60 rounds
Early stopping, best iteration is:
[126]	valid_0's binary_logloss: 0.580218
Training until validation scores don't improve for 60 rounds
Early stopping, best iteration is:
[3]	valid_0's binary_logloss: 0.592327
Training until validation scores don't improve fo

  0%|          | 0/80 [00:00<?, ?it/s]

Training until validation scores don't improve for 60 rounds
Early stopping, best iteration is:
[11]	valid_0's binary_logloss: 0.580053
Training until validation scores don't improve for 60 rounds
Early stopping, best iteration is:
[123]	valid_0's binary_logloss: 0.561706
Training until validation scores don't improve for 60 rounds
Early stopping, best iteration is:
[31]	valid_0's binary_logloss: 0.552466
Training until validation scores don't improve for 60 rounds
Early stopping, best iteration is:
[8]	valid_0's binary_logloss: 0.587689
Training until validation scores don't improve for 60 rounds
Early stopping, best iteration is:
[147]	valid_0's binary_logloss: 0.581554
Training until validation scores don't improve for 60 rounds
Early stopping, best iteration is:
[18]	valid_0's binary_logloss: 0.591101
Training until validation scores don't improve for 60 rounds
Early stopping, best iteration is:
[6]	valid_0's binary_logloss: 0.58266
Training until validation scores don't improve fo

In [2]:
"""
Step 3e — Two-stage Final Predictions
======================================
รวม Stage 1 (classifier) + Stage 2 (magnitude ensemble) เข้าด้วยกัน

Final prediction formula:
  prob     = P(demand > 0)  ← จาก Stage 1 classifier
  magnitude = ŷ_cat*w_cat + ŷ_lgb*w_lgb  ← Stage 2 stack (เดิม)
  ŷ_final  = prob × magnitude

ข้อดีของ soft combination (prob × magnitude) เทียบกับ hard threshold:
  - Conformal prediction intervals จะสะท้อน regime uncertainty ด้วย
  - ไม่มี discontinuity ที่ threshold → smooth predictions
  - ถ้า prob=0.3 และ magnitude=100,000 → final=30,000 (reasonable)

Input  : ml_features_phase4.csv
         catboost_models.pkl
         lightgbm_models.pkl
         stack_weights.pkl
         stage1_classifiers.pkl
         stage1_thresholds.pkl
Output : final_predictions_2stage.csv   ← พร้อมส่งต่อ Step 4 (Conformal)
         twostage_metrics.csv           ← MAE, NSE, KGE, zero-hit per (zone, h)
"""

import pandas as pd
import numpy as np
import joblib

from sklearn.metrics import mean_absolute_error, f1_score

# ── Config ────────────────────────────────────────────────────────────────────
HORIZON     = 12
ZONES       = ["zone_A", "zone_B"]
TRAIN_YEARS = [2020, 2021, 2022]
CALIB_YEAR  = 2023
TEST_YEAR   = 2024
ZERO_THRESH = 5000   # m³ — ต่ำกว่านี้นับว่า "predict zero"

CLASSIFIER_FEATURES = [
    "WoY_sin", "WoY_cos", "MoY_sin", "MoY_cos",
    "ET0_mm_week", "P_mm_week", "P_eff_mm",
    "SPI_4", "drought_flag", "MEI", "AI_week",
]
TARGET_LAGS = [1, 2, 3, 4]


def get_regressor_features(df: pd.DataFrame, df_zone: pd.DataFrame) -> list:
    exclude = {"year", "week", "month", "date", "zone", "target_col",
               "NIR_A_m3", "GIR_B_m3", "P_4week"}
    exclude |= {f"y_h{h}" for h in range(1, HORIZON + 1)}
    cols = [c for c in df.columns if c not in exclude]
    return [c for c in cols if not df_zone[c].isna().all()]


def get_clf_features(df_zone: pd.DataFrame, target_col: str) -> list:
    lag_cols  = [f"{target_col}_lag{k}" for k in TARGET_LAGS]
    roll_cols = [f"{target_col}_roll4_mean", f"{target_col}_roll8_mean"]
    wanted    = CLASSIFIER_FEATURES + lag_cols + roll_cols
    return [c for c in wanted
            if c in df_zone.columns and not df_zone[c].isna().all()]


def nse(y_obs, y_pred):
    denom = np.sum((y_obs - np.mean(y_obs))**2)
    return 1 - np.sum((y_obs - y_pred)**2) / denom if denom > 1e-6 else np.nan


def kge(y_obs, y_pred):
    r     = np.corrcoef(y_obs, y_pred)[0, 1] if len(y_obs) > 1 else np.nan
    alpha = np.std(y_pred) / np.std(y_obs) if np.std(y_obs) > 0 else np.nan
    beta  = np.mean(y_pred) / np.mean(y_obs) if np.mean(y_obs) > 0 else np.nan
    return 1 - np.sqrt((r-1)**2 + (alpha-1)**2 + (beta-1)**2)


def main():
    df          = pd.read_csv("ml_features_phase4.csv")
    cat_models  = joblib.load("catboost_models.pkl")
    lgb_models  = joblib.load("lightgbm_models.pkl")
    weights     = joblib.load("stack_weights.pkl")
    classifiers = joblib.load("stage1_classifiers.pkl")
    thresholds  = joblib.load("stage1_thresholds.pkl")

    all_preds = []
    metrics   = []

    for zone in ZONES:
        target_col  = "NIR_A_m3" if zone == "zone_A" else "GIR_B_m3"
        df_zone     = df[df["zone"] == zone].copy()
        reg_feats   = get_regressor_features(df, df_zone)
        clf_feats   = get_clf_features(df_zone, target_col)

        print(f"\n{'='*60}")
        print(f"Zone: {zone}")
        print(f"{'='*60}")

        # ── ทำนายทั้ง calib + test (เพื่อส่งต่อ conformal prediction) ────
        for split_year, split_name in [(CALIB_YEAR, "calib"), (TEST_YEAR, "test")]:
            split_df = df_zone[df_zone["year"] == split_year].copy()

            for h in range(1, HORIZON + 1):
                target_h = f"y_h{h}"
                valid = split_df.dropna(subset=reg_feats + clf_feats + [target_h])
                if len(valid) == 0:
                    continue

                X_reg = valid[reg_feats].values
                X_clf = valid[clf_feats].values
                y_obs = valid[target_h].values

                # Stage 1: probability of demand > 0
                clf   = classifiers[(zone, h)]
                prob  = clf.predict_proba(X_clf)[:, 1]

                # Stage 2: magnitude
                w     = weights[(zone, h)]
                mag   = (w["w_cat"] * cat_models[(zone,h)].predict(X_reg) +
                         w["w_lgb"] * lgb_models[(zone,h)].predict(X_reg))
                mag   = np.maximum(mag, 0)

                # Final: soft combination
                y_hat = prob * mag

                rows = valid[["year","week","month"]].copy()
                rows["zone"]      = zone
                rows["horizon"]   = h
                rows["split"]     = split_name
                rows["y_actual"]  = y_obs
                rows["y_stage1_prob"]   = prob.round(4)
                rows["y_stage2_mag"]    = mag.round(2)
                rows["y_pred_2stage"]   = y_hat.round(2)
                all_preds.append(rows)

                # Metrics for test set
                if split_name == "test":
                    mae_val   = mean_absolute_error(y_obs, y_hat)
                    nse_val   = nse(y_obs, y_hat)
                    kge_val   = kge(y_obs, y_hat)
                    # Wet season only
                    wet       = y_obs > 0
                    dry       = ~wet
                    mae_wet   = mean_absolute_error(y_obs[wet], y_hat[wet]) if wet.sum()>0 else np.nan
                    nse_wet   = nse(y_obs[wet], y_hat[wet]) if wet.sum()>1 else np.nan
                    mape_wet  = (np.abs((y_obs[wet]-y_hat[wet])/y_obs[wet]).mean()*100
                                 if wet.sum()>0 else np.nan)
                    zero_hit  = (y_hat[dry] < ZERO_THRESH).mean()*100 if dry.sum()>0 else np.nan
                    # F1 for binary regime
                    thr       = thresholds[(zone, h)]
                    y_bin_obs = (y_obs > 0).astype(int)
                    y_bin_hat = (y_hat > ZERO_THRESH).astype(int)
                    f1_val    = f1_score(y_bin_obs, y_bin_hat, zero_division=0)

                    metrics.append({
                        "zone": zone, "horizon": h,
                        "mae_overall":  round(mae_val,  2),
                        "nse_overall":  round(nse_val,  4),
                        "kge_overall":  round(kge_val,  4),
                        "mae_wet":      round(mae_wet,  2) if not np.isnan(mae_wet) else np.nan,
                        "nse_wet":      round(nse_wet,  4) if not np.isnan(nse_wet) else np.nan,
                        "mape_wet_pct": round(mape_wet, 1) if not np.isnan(mape_wet) else np.nan,
                        "zero_hit_pct": round(zero_hit, 1) if not np.isnan(zero_hit) else np.nan,
                        "f1_regime":    round(f1_val,   4),
                        "n_wet":        int(wet.sum()),
                        "n_dry":        int(dry.sum()),
                    })

                    print(f"  h={h:02d} | MAE={mae_val:,.0f}  NSE={nse_val:.3f}  "
                          f"NSE_wet={nse_wet:.3f}  MAPE_wet={mape_wet:.0f}%  "
                          f"zero-hit={zero_hit:.0f}%  F1={f1_val:.3f}")

    # ── Save ──────────────────────────────────────────────────────────────
    preds_df = pd.concat(all_preds, ignore_index=True)
    preds_df.to_csv("final_predictions_2stage.csv", index=False)

    metrics_df = pd.DataFrame(metrics)
    metrics_df.to_csv("twostage_metrics.csv", index=False)

    print(f"\n{'='*60}")
    print(f"Saved final_predictions_2stage.csv ({len(preds_df)} rows)")
    print(f"Saved twostage_metrics.csv")
    print(f"{'='*60}")

    # ── Summary ───────────────────────────────────────────────────────────
    for zone in ZONES:
        sub = metrics_df[metrics_df["zone"] == zone]
        print(f"\n{zone} — test 2024 summary:")
        print(f"  avg MAE (overall) = {sub.mae_overall.mean():,.0f} m³")
        print(f"  avg NSE (overall) = {sub.nse_overall.mean():.3f}")
        print(f"  avg NSE (wet)     = {sub.nse_wet.mean():.3f}")
        print(f"  avg MAPE (wet)    = {sub.mape_wet_pct.mean():.1f}%")
        print(f"  avg zero-hit      = {sub.zero_hit_pct.mean():.1f}%")
        print(f"  avg F1 (regime)   = {sub.f1_regime.mean():.3f}")

    return preds_df, metrics_df


if __name__ == "__main__":
    main()


Zone: zone_A
  h=01 | MAE=72,807  NSE=-0.110  NSE_wet=-0.441  MAPE_wet=21638%  zero-hit=0%  F1=0.828
  h=02 | MAE=76,674  NSE=-0.178  NSE_wet=-0.536  MAPE_wet=26215%  zero-hit=0%  F1=0.824
  h=03 | MAE=69,436  NSE=-0.091  NSE_wet=-0.428  MAPE_wet=14020%  zero-hit=0%  F1=0.819
  h=04 | MAE=69,785  NSE=-0.016  NSE_wet=-0.338  MAPE_wet=13075%  zero-hit=7%  F1=0.825
  h=05 | MAE=71,116  NSE=-0.001  NSE_wet=-0.315  MAPE_wet=33375%  zero-hit=0%  F1=0.810
  h=06 | MAE=72,488  NSE=-0.072  NSE_wet=-0.403  MAPE_wet=17744%  zero-hit=7%  F1=0.767
  h=07 | MAE=75,492  NSE=-0.076  NSE_wet=-0.408  MAPE_wet=44243%  zero-hit=0%  F1=0.800
  h=08 | MAE=76,314  NSE=-0.032  NSE_wet=-0.317  MAPE_wet=25138%  zero-hit=0%  F1=0.795
  h=09 | MAE=72,932  NSE=-0.012  NSE_wet=-0.294  MAPE_wet=19234%  zero-hit=0%  F1=0.754
  h=10 | MAE=70,246  NSE=-0.087  NSE_wet=-0.404  MAPE_wet=6141%  zero-hit=33%  F1=0.724
  h=11 | MAE=72,695  NSE=-0.026  NSE_wet=-0.314  MAPE_wet=17851%  zero-hit=0%  F1=0.776
  h=12 | MAE=77,25

In [3]:
"""
Step 4 — Conformal Prediction Intervals (Two-stage version)
============================================================
ปรับจาก step4_conformal.py เดิมให้รองรับ two-stage predictions

สิ่งที่เปลี่ยน vs เดิม:
  - ไม่ใช้ direct_models_all.pkl (ไม่มีแล้ว)
  - โหลด predictions จาก final_predictions_2stage.csv โดยตรง
    (calib 2023 + test 2024 อยู่ในไฟล์เดียวกันแล้ว)
  - nonconformity score = |y_actual - y_pred_2stage|
  - ที่เหลือ (q_hat, interval, coverage) เหมือนเดิม

Input  : final_predictions_2stage.csv   ← จาก step3e
Output : forecast_conformal_2stage.csv
         conformal_coverage_summary.csv

Run    : python step4_conformal_2stage.py
"""

import pandas as pd
import numpy as np

HORIZON    = 12
ZONES      = ["zone_A", "zone_B"]
ALPHA      = 0.10        # target coverage = 90%
CALIB_YEAR = 2023
TEST_YEAR  = 2024


def conformal_quantile(scores: np.ndarray, alpha: float) -> float:
    """
    Finite-sample corrected conformal quantile.
    q̂ = quantile(scores, ceil((n+1)(1-α)) / n)
    """
    n       = len(scores)
    q_level = min(np.ceil((n + 1) * (1 - alpha)) / n, 1.0)
    return float(np.quantile(scores, q_level))


def main():
    preds = pd.read_csv("final_predictions_2stage.csv")

    # ── ตรวจ columns ──────────────────────────────────────────────────────
    required = {"year", "week", "zone", "horizon", "split",
                "y_actual", "y_pred_2stage"}
    missing  = required - set(preds.columns)
    if missing:
        raise KeyError(f"Missing columns in final_predictions_2stage.csv: {missing}")

    all_results    = []
    coverage_table = {}

    for zone in ZONES:
        z_preds = preds[preds["zone"] == zone].copy()

        calib_df = z_preds[z_preds["split"] == "calib"]
        test_df  = z_preds[z_preds["split"] == "test"]

        print(f"\n{'='*55}")
        print(f"Zone: {zone}  |  "
              f"calib={len(calib_df)} rows  test={len(test_df)} rows")
        print(f"{'='*55}")

        for h in range(1, HORIZON + 1):
            # ── Calibration nonconformity scores (2023) ───────────────────
            calib_h = calib_df[calib_df["horizon"] == h].dropna(
                subset=["y_actual", "y_pred_2stage"])
            y_ca    = calib_h["y_actual"].values
            yhat_ca = calib_h["y_pred_2stage"].values
            scores  = np.abs(y_ca - yhat_ca)

            if len(scores) == 0:
                print(f"  h={h:02d} | ⚠ no calibration rows — skipped")
                continue

            q_hat = conformal_quantile(scores, ALPHA)

            # ── Test predictions + intervals (2024) ───────────────────────
            test_h  = test_df[test_df["horizon"] == h].dropna(
                subset=["y_actual", "y_pred_2stage"])
            y_te    = test_h["y_actual"].values
            yhat_te = test_h["y_pred_2stage"].values

            lower = np.maximum(yhat_te - q_hat, 0)   # demand ≥ 0
            upper = yhat_te + q_hat

            # ── Coverage & interval width ─────────────────────────────────
            covered  = (y_te >= lower) & (y_te <= upper)
            coverage = covered.mean()
            iw_mean  = (upper - lower).mean()
            iw_med   = np.median(upper - lower)

            # Conditional coverage: wet vs dry
            wet      = y_te > 0
            dry      = ~wet
            cov_wet  = covered[wet].mean()  if wet.sum()  > 0 else np.nan
            cov_dry  = covered[dry].mean()  if dry.sum()  > 0 else np.nan

            coverage_table[(zone, h)] = {
                "coverage":    coverage,
                "cov_wet":     cov_wet,
                "cov_dry":     cov_dry,
                "q_hat":       q_hat,
                "IW_mean":     iw_mean,
                "IW_median":   iw_med,
                "n_calib":     len(scores),
                "n_test":      len(y_te),
            }

            print(f"  h={h:02d} | q̂={q_hat:>9,.0f} m³ | "
                  f"coverage={coverage:.3f} "
                  f"(wet={cov_wet:.2f} dry={cov_dry:.2f}) | "
                  f"IW={iw_mean:,.0f} m³")

            # ── Collect rows ──────────────────────────────────────────────
            out = test_h[["year","week","zone","horizon",
                           "y_actual","y_stage1_prob",
                           "y_stage2_mag","y_pred_2stage"]].copy()
            out["lower_90"] = lower
            out["upper_90"] = upper
            out["q_hat"]    = q_hat
            out["covered"]  = covered.astype(int)
            all_results.append(out)

    # ── Save predictions ──────────────────────────────────────────────────
    forecast_df = pd.concat(all_results, ignore_index=True)
    forecast_df.to_csv("forecast_conformal_2stage.csv", index=False)
    print(f"\nSaved forecast_conformal_2stage.csv — {len(forecast_df)} rows")

    # ── Coverage summary ──────────────────────────────────────────────────
    cov_rows = [{"zone": z, "horizon": h, **stats}
                for (z, h), stats in coverage_table.items()]
    cov_df = pd.DataFrame(cov_rows)
    cov_df.to_csv("conformal_coverage_summary.csv", index=False)
    print("Saved conformal_coverage_summary.csv")

    # ── Print tables ──────────────────────────────────────────────────────
    print(f"\n{'='*55}")
    print(f"Coverage (target: {1-ALPHA:.0%})  — below {1-ALPHA:.0%} = under-covered")
    print(f"{'='*55}")
    piv_cov = cov_df.pivot(index="horizon", columns="zone",
                            values="coverage").round(3)
    piv_wet = cov_df.pivot(index="horizon", columns="zone",
                            values="cov_wet").round(3)
    piv_dry = cov_df.pivot(index="horizon", columns="zone",
                            values="cov_dry").round(3)
    piv_iw  = cov_df.pivot(index="horizon", columns="zone",
                            values="IW_mean").round(0)
    piv_qh  = cov_df.pivot(index="horizon", columns="zone",
                            values="q_hat").round(0)

    print("\nOverall coverage:")
    print(piv_cov.to_string())
    print("\nWet-season coverage (demand > 0):")
    print(piv_wet.to_string())
    print("\nDry-season coverage (demand = 0):")
    print(piv_dry.to_string())
    print("\nMean Interval Width (m³/week):")
    print(piv_iw.to_string())
    print("\nq̂ (nonconformity quantile, m³):")
    print(piv_qh.to_string())

    # ── Aggregate ─────────────────────────────────────────────────────────
    print(f"\n{'='*55}")
    print("Aggregate summary")
    print(f"{'='*55}")
    for zone in ZONES:
        sub = cov_df[cov_df["zone"] == zone]
        below = (sub["coverage"] < (1 - ALPHA)).sum()
        print(f"\n{zone}:")
        print(f"  avg coverage = {sub['coverage'].mean():.3f}  "
              f"(target {1-ALPHA:.0%})")
        print(f"  below target = {below}/12 horizons")
        print(f"  avg IW       = {sub['IW_mean'].mean():,.0f} m³/week")
        print(f"  avg q̂       = {sub['q_hat'].mean():,.0f} m³")

    return forecast_df, cov_df


if __name__ == "__main__":
    main()


Zone: zone_A  |  calib=624 rows  test=546 rows
  h=01 | q̂=  293,310 m³ | coverage=0.941 (wet=0.92 dry=1.00) | IW=331,368 m³
  h=02 | q̂=  285,820 m³ | coverage=0.920 (wet=0.89 dry=1.00) | IW=322,074 m³
  h=03 | q̂=  284,959 m³ | coverage=0.939 (wet=0.91 dry=1.00) | IW=319,871 m³
  h=04 | q̂=  281,655 m³ | coverage=0.938 (wet=0.91 dry=1.00) | IW=318,460 m³
  h=05 | q̂=  291,277 m³ | coverage=0.936 (wet=0.91 dry=1.00) | IW=329,729 m³
  h=06 | q̂=  295,592 m³ | coverage=0.935 (wet=0.90 dry=1.00) | IW=332,605 m³
  h=07 | q̂=  286,356 m³ | coverage=0.933 (wet=0.90 dry=1.00) | IW=323,973 m³
  h=08 | q̂=  287,348 m³ | coverage=0.932 (wet=0.90 dry=1.00) | IW=325,654 m³
  h=09 | q̂=  290,824 m³ | coverage=0.930 (wet=0.89 dry=1.00) | IW=324,830 m³
  h=10 | q̂=  301,569 m³ | coverage=0.929 (wet=0.89 dry=1.00) | IW=327,671 m³
  h=11 | q̂=  293,333 m³ | coverage=0.927 (wet=0.88 dry=1.00) | IW=323,219 m³
  h=12 | q̂=  296,266 m³ | coverage=0.925 (wet=0.88 dry=1.00) | IW=333,246 m³

Zone: zone_B  |

In [6]:
"""
Step 4v2 — Mondrian Conformal Prediction (Regime-conditional)
=============================================================
แทนที่ global q̂ ด้วย regime-specific q̂:

  q̂_wet  = conformal_quantile(scores[y_calib > 0],  α)
  q̂_dry  = conformal_quantile(scores[y_calib == 0], α)

Apply ตาม Stage 1 probability:
  prob >= threshold → apply q̂_wet  (active regime)
  prob <  threshold → apply q̂_dry  (inactive regime)

ข้อดีทางทฤษฎี (Venn–Mondrian conformal):
  - valid marginal coverage ภายใต้ exchangeability ต่อ regime
  - narrower intervals เพราะ calibrate เฉพาะ within-regime residuals
  - ไม่มี dry-season transition outliers ดึง q̂_wet ขึ้น

Input  : final_predictions_2stage.csv
         stage1_thresholds.pkl
Output : forecast_mondrian.csv
         mondrian_coverage_summary.csv

Run: python step4v2_mondrian_conformal.py
"""

import pandas as pd
import numpy as np
import joblib

HORIZON    = 12
ZONES      = ["zone_A", "zone_B"]
ALPHA      = 0.10           # target coverage = 90%
CALIB_YEAR = 2023
TEST_YEAR  = 2024
MIN_REGIME_N = 5            # ถ้า regime มีน้อยกว่านี้ → fallback to global q̂


def conformal_quantile(scores: np.ndarray, alpha: float) -> float:
    """Finite-sample corrected conformal quantile."""
    n = len(scores)
    if n == 0:
        return np.inf
    q_level = min(np.ceil((n + 1) * (1 - alpha)) / n, 1.0)
    return float(np.quantile(scores, q_level))


def main():
    preds      = pd.read_csv("final_predictions_2stage.csv")
    thresholds = joblib.load("stage1_thresholds.pkl")

    all_results    = []
    coverage_table = {}

    for zone in ZONES:
        z = preds[preds["zone"] == zone].copy()
        calib_df = z[z["split"] == "calib"]
        test_df  = z[z["split"] == "test"]

        print(f"\n{'='*60}")
        print(f"Zone: {zone}  |  calib={len(calib_df)}  test={len(test_df)}")
        print(f"{'='*60}")

        for h in range(1, HORIZON + 1):
            thr = thresholds[(zone, h)]

            # ── Calibration split by regime ───────────────────────────────
            ca_h = calib_df[calib_df["horizon"] == h].dropna(
                subset=["y_actual", "y_pred_2stage", "y_stage1_prob"])

            wet_mask_ca = ca_h["y_actual"] > 0
            dry_mask_ca = ~wet_mask_ca

            scores_all = np.abs(ca_h["y_actual"] - ca_h["y_pred_2stage"])
            scores_wet = scores_all[wet_mask_ca].values
            scores_dry = scores_all[dry_mask_ca].values

            n_wet_ca = len(scores_wet)
            n_dry_ca = len(scores_dry)

            # Compute regime-specific q̂ (fallback to global if too few)
            q_hat_global = conformal_quantile(scores_all.values, ALPHA)
            q_hat_wet    = (conformal_quantile(scores_wet, ALPHA)
                            if n_wet_ca >= MIN_REGIME_N else q_hat_global)
            q_hat_dry    = (conformal_quantile(scores_dry, ALPHA)
                            if n_dry_ca >= MIN_REGIME_N else q_hat_global)

            # ── Test predictions ──────────────────────────────────────────
            te_h = test_df[test_df["horizon"] == h].dropna(
                subset=["y_actual", "y_pred_2stage", "y_stage1_prob"])

            y_te    = te_h["y_actual"].values
            yhat_te = te_h["y_pred_2stage"].values
            prob_te = te_h["y_stage1_prob"].values

            # Assign q̂ per row based on Stage 1 classification
            active_mask = prob_te >= thr
            q_applied   = np.where(active_mask, q_hat_wet, q_hat_dry)

            lower = np.maximum(yhat_te - q_applied, 0)
            upper = yhat_te + q_applied

            # ── Coverage metrics ──────────────────────────────────────────
            covered  = (y_te >= lower) & (y_te <= upper)
            coverage = covered.mean()
            iw       = (upper - lower).mean()
            iw_med   = np.median(upper - lower)

            wet_te   = y_te > 0
            dry_te   = ~wet_te
            cov_wet  = covered[wet_te].mean() if wet_te.sum() > 0 else np.nan
            cov_dry  = covered[dry_te].mean() if dry_te.sum() > 0 else np.nan

            # vs global (reference)
            lower_g  = np.maximum(yhat_te - q_hat_global, 0)
            upper_g  = yhat_te + q_hat_global
            cov_g    = ((y_te >= lower_g) & (y_te <= upper_g)).mean()
            iw_g     = (upper_g - lower_g).mean()

            coverage_table[(zone, h)] = {
                "coverage":     coverage,
                "cov_wet":      cov_wet,
                "cov_dry":      cov_dry,
                "cov_global":   cov_g,
                "q_hat_wet":    q_hat_wet,
                "q_hat_dry":    q_hat_dry,
                "q_hat_global": q_hat_global,
                "IW_mondrian":  iw,
                "IW_global":    iw_g,
                "IW_median":    iw_med,
                "n_calib_wet":  n_wet_ca,
                "n_calib_dry":  n_dry_ca,
                "n_test":       len(y_te),
                "threshold":    thr,
            }

            delta_iw  = iw - iw_g
            delta_cov = coverage - cov_g
            print(f"  h={h:02d} | q̂_wet={q_hat_wet:>8,.0f}  "
                  f"q̂_dry={q_hat_dry:>8,.0f}  "
                  f"cov={coverage:.3f} (wet={cov_wet:.2f} dry={cov_dry:.2f})  "
                  f"IW={iw:,.0f}  "
                  f"Δcov={delta_cov:+.3f}  ΔIW={delta_iw:+,.0f}")

            # ── Collect output rows ───────────────────────────────────────
            out = te_h[["year","week","zone","horizon",
                         "y_actual","y_stage1_prob",
                         "y_stage2_mag","y_pred_2stage"]].copy()
            out["lower_90_mondrian"] = lower
            out["upper_90_mondrian"] = upper
            out["lower_90_global"]   = lower_g
            out["upper_90_global"]   = upper_g
            out["q_hat_applied"]     = q_applied
            out["q_hat_wet"]         = q_hat_wet
            out["q_hat_dry"]         = q_hat_dry
            out["regime_active"]     = active_mask.astype(int)
            out["covered_mondrian"]  = covered.astype(int)
            out["covered_global"]    = ((y_te >= lower_g) &
                                        (y_te <= upper_g)).astype(int)
            all_results.append(out)

    # ── Save ──────────────────────────────────────────────────────────────
    forecast_df = pd.concat(all_results, ignore_index=True)
    forecast_df.to_csv("forecast_mondrian.csv", index=False)

    cov_rows = [{"zone": z, "horizon": h, **stats}
                for (z, h), stats in coverage_table.items()]
    cov_df = pd.DataFrame(cov_rows)
    cov_df.to_csv("mondrian_coverage_summary.csv", index=False)

    print(f"\nSaved forecast_mondrian.csv — {len(forecast_df)} rows")
    print("Saved mondrian_coverage_summary.csv")

    # ── Summary tables ────────────────────────────────────────────────────
    print(f"\n{'='*60}")
    print(f"Mondrian vs Global — coverage (target {1-ALPHA:.0%})")
    print(f"{'='*60}")
    piv_mon  = cov_df.pivot(index="horizon", columns="zone", values="coverage").round(3)
    piv_glo  = cov_df.pivot(index="horizon", columns="zone", values="cov_global").round(3)
    piv_wet  = cov_df.pivot(index="horizon", columns="zone", values="cov_wet").round(3)
    piv_iw_m = cov_df.pivot(index="horizon", columns="zone", values="IW_mondrian").round(0)
    piv_iw_g = cov_df.pivot(index="horizon", columns="zone", values="IW_global").round(0)

    print("\nOverall coverage — Mondrian:")
    print(piv_mon.to_string())
    print("\nOverall coverage — Global (reference):")
    print(piv_glo.to_string())
    print("\nWet-season coverage — Mondrian:")
    print(piv_wet.to_string())
    print("\nMean IW — Mondrian (m³):")
    print(piv_iw_m.to_string())
    print("\nMean IW — Global (m³):")
    print(piv_iw_g.to_string())

    # ── Aggregate ─────────────────────────────────────────────────────────
    print(f"\n{'='*60}")
    print("Aggregate comparison: Mondrian vs Global")
    print(f"{'='*60}")
    for zone in ZONES:
        mean_demand = 85810 if zone == "zone_A" else 30187
        sub = cov_df[cov_df["zone"] == zone]
        print(f"\n{zone}:")
        print(f"  coverage  Mon={sub.coverage.mean():.3f}  "
              f"Glo={sub.cov_global.mean():.3f}  "
              f"Δ={sub.coverage.mean()-sub.cov_global.mean():+.3f}")
        print(f"  below 90% Mon={( sub.coverage<0.9).sum()}/12  "
              f"Glo={(sub.cov_global<0.9).sum()}/12")
        print(f"  IW_mean   Mon={sub.IW_mondrian.mean():,.0f}  "
              f"Glo={sub.IW_global.mean():,.0f}  "
              f"Δ={sub.IW_mondrian.mean()-sub.IW_global.mean():+,.0f} m³")
        print(f"  IW/demand Mon={sub.IW_mondrian.mean()/mean_demand:.2f}×  "
              f"Glo={sub.IW_global.mean()/mean_demand:.2f}×")
        print(f"  q̂_wet avg={sub.q_hat_wet.mean():,.0f}  "
              f"q̂_dry avg={sub.q_hat_dry.mean():,.0f}  "
              f"q̂_global={sub.q_hat_global.mean():,.0f}")

    return forecast_df, cov_df


if __name__ == "__main__":
    main()


Zone: zone_A  |  calib=624  test=546
  h=01 | q̂_wet= 310,386  q̂_dry=  59,726  cov=0.941 (wet=0.92 dry=1.00)  IW=333,699  Δcov=+0.000  ΔIW=+2,331
  h=02 | q̂_wet= 306,764  q̂_dry=  57,762  cov=0.920 (wet=0.89 dry=1.00)  IW=303,178  Δcov=+0.000  ΔIW=-18,896
  h=03 | q̂_wet= 320,607  q̂_dry=  40,088  cov=0.918 (wet=0.88 dry=1.00)  IW=298,270  Δcov=-0.020  ΔIW=-21,601
  h=04 | q̂_wet= 298,646  q̂_dry=  81,780  cov=0.938 (wet=0.91 dry=1.00)  IW=312,861  Δcov=+0.000  ΔIW=-5,599
  h=05 | q̂_wet= 299,429  q̂_dry=  55,902  cov=0.936 (wet=0.91 dry=1.00)  IW=332,700  Δcov=+0.000  ΔIW=+2,971
  h=06 | q̂_wet= 317,365  q̂_dry=  60,473  cov=0.935 (wet=0.90 dry=1.00)  IW=337,625  Δcov=+0.000  ΔIW=+5,019
  h=07 | q̂_wet= 297,859  q̂_dry=  65,387  cov=0.933 (wet=0.90 dry=1.00)  IW=314,812  Δcov=+0.000  ΔIW=-9,161
  h=08 | q̂_wet= 294,041  q̂_dry=  57,728  cov=0.909 (wet=0.90 dry=0.93)  IW=278,364  Δcov=-0.023  ΔIW=-47,289
  h=09 | q̂_wet= 307,624  q̂_dry= 106,720  cov=0.930 (wet=0.89 dry=1.00)  IW=31

In [7]:
"""
Step 4v3 — Normalized Mondrian Conformal Prediction
=====================================================
Combined approach:
  1. Normalized score  : s = |y − ŷ| / (|ŷ| + ε)
  2. Mondrian          : แยก q̂ ตาม regime (wet / dry)

Interval:  ŷ ± q̂_norm × (|ŷ| + ε)
  → กว้างเมื่อ prediction สูง  (wet season peaks)
  → แคบเมื่อ prediction ต่ำ   (dry season / transitions)
  → IW ปรับตาม magnitude อัตโนมัติ

Compare 4 variants ใน run เดียว:
  A. Global   + Absolute   (baseline — step4 เดิม)
  B. Mondrian + Absolute   (step4v2)
  C. Global   + Normalized (ใหม่)
  D. Mondrian + Normalized (ใหม่ — strongest)

Input  : final_predictions_2stage.csv
         stage1_thresholds.pkl
Output : forecast_normalized_mondrian.csv
         normalized_mondrian_summary.csv  (4-variant comparison)

Run: python step4v3_normalized_mondrian.py
"""

import pandas as pd
import numpy as np
import joblib

HORIZON      = 12
ZONES        = ["zone_A", "zone_B"]
ALPHA        = 0.10        # target coverage = 90%
CALIB_YEAR   = 2023
TEST_YEAR    = 2024
MIN_N        = 5           # min regime samples สำหรับ Mondrian
EPSILON      = 1_000       # m³ — ป้องกัน div/0 เมื่อ ŷ ≈ 0


def conformal_quantile(scores: np.ndarray, alpha: float) -> float:
    n = len(scores)
    if n == 0:
        return np.inf
    q_level = min(np.ceil((n + 1) * (1 - alpha)) / n, 1.0)
    return float(np.quantile(scores, q_level))


def abs_scores(y, yhat):
    return np.abs(y - yhat)


def norm_scores(y, yhat, eps=EPSILON):
    return np.abs(y - yhat) / (np.abs(yhat) + eps)


def apply_abs_interval(yhat, q):
    lower = np.maximum(yhat - q, 0)
    upper = yhat + q
    return lower, upper


def apply_norm_interval(yhat, q_norm, eps=EPSILON):
    half  = q_norm * (np.abs(yhat) + eps)
    lower = np.maximum(yhat - half, 0)
    upper = yhat + half
    return lower, upper


def coverage_stats(y, lower, upper):
    covered  = (y >= lower) & (y <= upper)
    wet      = y > 0
    dry      = ~wet
    return {
        "coverage":  covered.mean(),
        "cov_wet":   covered[wet].mean() if wet.sum() > 0 else np.nan,
        "cov_dry":   covered[dry].mean() if dry.sum() > 0 else np.nan,
        "IW_mean":   (upper - lower).mean(),
        "IW_median": np.median(upper - lower),
        "IW_wet":    (upper - lower)[wet].mean() if wet.sum() > 0 else np.nan,
    }


def main():
    preds      = pd.read_csv("final_predictions_2stage.csv")
    thresholds = joblib.load("stage1_thresholds.pkl")

    all_rows  = []
    log_rows  = []

    for zone in ZONES:
        mean_demand = 85_810 if zone == "zone_A" else 30_187
        z        = preds[preds["zone"] == zone].copy()
        calib_df = z[z["split"] == "calib"]
        test_df  = z[z["split"] == "test"]

        print(f"\n{'='*65}")
        print(f"Zone: {zone}  |  mean demand ≈ {mean_demand:,.0f} m³")
        print(f"{'='*65}")
        print(f"{'h':>3} | {'q̂abs_g':>10} {'q̂abs_wet':>10} {'q̂nrm_g':>8} {'q̂nrm_wet':>9} "
              f"| {'covA':>5} {'covB':>5} {'covC':>5} {'covD':>5} "
              f"| {'IW_A':>8} {'IW_D':>8} {'ratio':>5}")
        print("-"*100)

        for h in range(1, HORIZON + 1):
            thr   = thresholds[(zone, h)]
            ca_h  = calib_df[calib_df["horizon"] == h].dropna(
                        subset=["y_actual","y_pred_2stage","y_stage1_prob"])
            te_h  = test_df[test_df["horizon"] == h].dropna(
                        subset=["y_actual","y_pred_2stage","y_stage1_prob"])

            y_ca   = ca_h["y_actual"].values
            yh_ca  = ca_h["y_pred_2stage"].values
            y_te   = te_h["y_actual"].values
            yh_te  = te_h["y_pred_2stage"].values
            prob   = te_h["y_stage1_prob"].values
            active = prob >= thr

            wet_ca = y_ca > 0
            dry_ca = ~wet_ca

            # ── Calibration scores ────────────────────────────────────────
            s_abs_all  = abs_scores(y_ca, yh_ca)
            s_abs_wet  = s_abs_all[wet_ca]
            s_abs_dry  = s_abs_all[dry_ca]

            s_nrm_all  = norm_scores(y_ca, yh_ca)
            s_nrm_wet  = s_nrm_all[wet_ca]
            s_nrm_dry  = s_nrm_all[dry_ca]

            # ── q̂ per variant ─────────────────────────────────────────────
            q_abs_g   = conformal_quantile(s_abs_all, ALPHA)
            q_abs_wet = (conformal_quantile(s_abs_wet, ALPHA)
                         if len(s_abs_wet) >= MIN_N else q_abs_g)
            q_abs_dry = (conformal_quantile(s_abs_dry, ALPHA)
                         if len(s_abs_dry) >= MIN_N else q_abs_g)

            q_nrm_g   = conformal_quantile(s_nrm_all, ALPHA)
            q_nrm_wet = (conformal_quantile(s_nrm_wet, ALPHA)
                         if len(s_nrm_wet) >= MIN_N else q_nrm_g)
            q_nrm_dry = (conformal_quantile(s_nrm_dry, ALPHA)
                         if len(s_nrm_dry) >= MIN_N else q_nrm_g)

            # ── Test intervals ────────────────────────────────────────────
            # A: Global + Absolute
            lo_A, up_A = apply_abs_interval(yh_te, q_abs_g)
            # B: Mondrian + Absolute
            q_abs_app  = np.where(active, q_abs_wet, q_abs_dry)
            lo_B, up_B = apply_abs_interval(yh_te, q_abs_app)
            # C: Global + Normalized
            lo_C, up_C = apply_norm_interval(yh_te, q_nrm_g)
            # D: Mondrian + Normalized  ← primary output
            q_nrm_app  = np.where(active, q_nrm_wet, q_nrm_dry)
            lo_D, up_D = apply_norm_interval(yh_te, q_nrm_app)

            # ── Coverage & IW ─────────────────────────────────────────────
            covA = coverage_stats(y_te, lo_A, up_A)
            covB = coverage_stats(y_te, lo_B, up_B)
            covC = coverage_stats(y_te, lo_C, up_C)
            covD = coverage_stats(y_te, lo_D, up_D)

            iw_ratio = covD["IW_mean"] / mean_demand

            print(f"  {h:2d} | "
                  f"{q_abs_g:>10,.0f} {q_abs_wet:>10,.0f} "
                  f"{q_nrm_g:>8.3f} {q_nrm_wet:>9.3f} | "
                  f"{covA['coverage']:>5.3f} {covB['coverage']:>5.3f} "
                  f"{covC['coverage']:>5.3f} {covD['coverage']:>5.3f} | "
                  f"{covA['IW_mean']:>8,.0f} {covD['IW_mean']:>8,.0f} "
                  f"{iw_ratio:>5.2f}×")

            log_rows.append({
                "zone": zone, "horizon": h,
                "threshold": thr,
                # q̂ values
                "q_abs_global":   round(q_abs_g,   2),
                "q_abs_wet":      round(q_abs_wet,  2),
                "q_abs_dry":      round(q_abs_dry,  2),
                "q_norm_global":  round(q_nrm_g,    6),
                "q_norm_wet":     round(q_nrm_wet,  6),
                "q_norm_dry":     round(q_nrm_dry,  6),
                # Coverage — A
                "cov_A":     round(covA["coverage"], 4),
                "cov_wet_A": round(covA["cov_wet"],  4) if not np.isnan(covA["cov_wet"]) else np.nan,
                "IW_A":      round(covA["IW_mean"],  2),
                # Coverage — B
                "cov_B":     round(covB["coverage"], 4),
                "cov_wet_B": round(covB["cov_wet"],  4) if not np.isnan(covB["cov_wet"]) else np.nan,
                "IW_B":      round(covB["IW_mean"],  2),
                # Coverage — C
                "cov_C":     round(covC["coverage"], 4),
                "cov_wet_C": round(covC["cov_wet"],  4) if not np.isnan(covC["cov_wet"]) else np.nan,
                "IW_C":      round(covC["IW_mean"],  2),
                # Coverage — D (primary)
                "cov_D":        round(covD["coverage"], 4),
                "cov_wet_D":    round(covD["cov_wet"],  4) if not np.isnan(covD["cov_wet"]) else np.nan,
                "cov_dry_D":    round(covD["cov_dry"],  4) if not np.isnan(covD["cov_dry"]) else np.nan,
                "IW_D":         round(covD["IW_mean"],  2),
                "IW_wet_D":     round(covD["IW_wet"],   2) if not np.isnan(covD["IW_wet"]) else np.nan,
                "IW_ratio_D":   round(iw_ratio, 3),
                "n_calib_wet":  int(wet_ca.sum()),
                "n_calib_dry":  int(dry_ca.sum()),
                "n_test":       len(y_te),
            })

            # ── Collect test rows (variant D as primary) ──────────────────
            out = te_h[["year","week","zone","horizon",
                         "y_actual","y_stage1_prob",
                         "y_stage2_mag","y_pred_2stage"]].copy()
            out["lower_90"]       = lo_D
            out["upper_90"]       = up_D
            out["lower_90_A"]     = lo_A
            out["upper_90_A"]     = up_A
            out["q_norm_applied"] = q_nrm_app
            out["regime_active"]  = active.astype(int)
            out["covered_D"]      = ((y_te >= lo_D) & (y_te <= up_D)).astype(int)
            out["covered_A"]      = ((y_te >= lo_A) & (y_te <= up_A)).astype(int)
            all_rows.append(out)

    # ── Save ──────────────────────────────────────────────────────────────
    forecast_df = pd.concat(all_rows, ignore_index=True)
    forecast_df.to_csv("forecast_normalized_mondrian.csv", index=False)

    summary = pd.DataFrame(log_rows)
    summary.to_csv("normalized_mondrian_summary.csv", index=False)

    print(f"\nSaved forecast_normalized_mondrian.csv — {len(forecast_df)} rows")
    print("Saved normalized_mondrian_summary.csv")

    # ── Aggregate comparison ──────────────────────────────────────────────
    print(f"\n{'='*65}")
    print("AGGREGATE COMPARISON — 4 variants (target coverage 90%)")
    print(f"{'='*65}")
    print(f"{'':8} {'A:Glo+Abs':>12} {'B:Mon+Abs':>12} "
          f"{'C:Glo+Nrm':>12} {'D:Mon+Nrm':>12}")
    print("-"*60)

    for zone in ZONES:
        mean_dem = 85_810 if zone == "zone_A" else 30_187
        sub = summary[summary["zone"] == zone]
        for metric, cols, fmt in [
            ("coverage",  ["cov_A","cov_B","cov_C","cov_D"],  ".3f"),
            ("below 90%", None, None),
            ("IW_mean",   ["IW_A","IW_B","IW_C","IW_D"],      ",.0f"),
            ("IW/demand", None, None),
        ]:
            if metric == "below 90%":
                vals = [f"{(sub[c]<0.9).sum()}/12" for c in ["cov_A","cov_B","cov_C","cov_D"]]
                print(f"  {zone[:6]:8} {metric:10} "+" ".join(f"{v:>12}" for v in vals))
            elif metric == "IW/demand":
                vals = [f"{sub[c].mean()/mean_dem:.2f}×" for c in ["IW_A","IW_B","IW_C","IW_D"]]
                print(f"  {'':8} {'IW/demand':10} "+" ".join(f"{v:>12}" for v in vals))
                print()
            else:
                vals = [f"{sub[c].mean():{fmt}}" for c in cols]
                print(f"  {zone[:6]:8} {metric:10} "+" ".join(f"{v:>12}" for v in vals))


if __name__ == "__main__":
    main()


Zone: zone_A  |  mean demand ≈ 85,810 m³
  h |    q̂abs_g  q̂abs_wet  q̂nrm_g q̂nrm_wet |  covA  covB  covC  covD |     IW_A     IW_D ratio
----------------------------------------------------------------------------------------------------
   1 |    293,310    310,386   13.621    14.275 | 0.941 0.941 0.922 0.961 |  331,368  577,970  6.74×
   2 |    285,820    306,764   11.124    12.427 | 0.920 0.920 0.960 0.940 |  322,074  456,038  5.31×
   3 |    284,959    320,607   11.781    12.489 | 0.939 0.918 0.959 0.918 |  319,871  452,828  5.28×
   4 |    281,655    298,646    8.649     8.912 | 0.938 0.938 0.938 0.938 |  318,460  360,099  4.20×
   5 |    291,277    299,429    8.728     8.842 | 0.936 0.936 0.936 0.936 |  329,729  384,428  4.48×
   6 |    295,592    317,365   12.381    16.958 | 0.935 0.935 0.935 0.935 |  332,605  673,194  7.85×
   7 |    286,356    297,859   13.912    14.559 | 0.933 0.933 0.956 0.956 |  323,973  587,519  6.85×
   8 |    287,348    294,041   11.940    12.687 | 0

In [2]:
"""
Step 5 — Performance Metrics + SHAP Analysis (Two-stage version)
=================================================================
ปรับจาก step5_metrics_shap.py เดิม:
  - ไม่ใช้ direct_models_all.pkl (ไม่มีแล้ว)
  - โหลด catboost_models.pkl / lightgbm_models.pkl / stack_weights.pkl
  - metrics จาก forecast_mondrian.csv (Variant B — final output)
  - เพิ่ม Stage 1 classifier SHAP
  - แก้ persistence baseline: GID_B_m3 → GIR_B_m3
  - เพิ่ม wet/dry season metrics breakdown

Input  : ml_features_phase4.csv
         forecast_mondrian.csv         ← Step 4v2 output
         catboost_models.pkl
         stage1_classifiers.pkl
Output : performance_metrics_all.csv
         shap_beeswarm_zone_A/B.png
         shap_importance_zone_A/B.csv
         shap_by_horizon_zone_A/B.csv
         shap_stage1_zone_A/B.png      ← classifier SHAP (ใหม่)
"""

import pandas as pd
import numpy as np
import joblib
import shap
import warnings
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
warnings.filterwarnings("ignore")

HORIZON   = 12
ZONES     = ["zone_A", "zone_B"]
TEST_YEAR = 2024

CLASSIFIER_FEATURES = [
    "WoY_sin", "WoY_cos", "MoY_sin", "MoY_cos",
    "ET0_mm_week", "P_mm_week", "P_eff_mm",
    "SPI_4", "drought_flag", "MEI", "AI_week",
]
TARGET_LAGS = [1, 2, 3, 4]


# ─────────────────────────────────────────────────────────────────────────────
#  Helpers
# ─────────────────────────────────────────────────────────────────────────────
def get_regressor_features(df: pd.DataFrame,
                            df_zone: pd.DataFrame) -> list:
    exclude = {"year", "week", "month", "date", "zone", "target_col",
               "NIR_A_m3", "GIR_B_m3", "P_4week"}
    exclude |= {f"y_h{h}" for h in range(1, HORIZON + 1)}
    cols = [c for c in df.columns if c not in exclude]
    return [c for c in cols if not df_zone[c].isna().all()]


def get_clf_features(df_zone: pd.DataFrame, target_col: str) -> list:
    lag_cols  = [f"{target_col}_lag{k}" for k in TARGET_LAGS]
    roll_cols = [f"{target_col}_roll4_mean", f"{target_col}_roll8_mean"]
    wanted    = CLASSIFIER_FEATURES + lag_cols + roll_cols
    return [c for c in wanted
            if c in df_zone.columns and not df_zone[c].isna().all()]


def calc_metrics(y_obs: np.ndarray, y_sim: np.ndarray,
                 label: str = "Model") -> dict:
    eps  = 1e-6
    mae  = np.mean(np.abs(y_obs - y_sim))
    rmse = np.sqrt(np.mean((y_obs - y_sim) ** 2))
    # MAPE nonzero only (avoid astronomical values from near-zero weeks)
    nz   = y_obs > 0
    mape = (np.mean(np.abs((y_obs[nz] - y_sim[nz]) / y_obs[nz])) * 100
            if nz.sum() > 0 else np.nan)
    sst  = np.sum((y_obs - np.mean(y_obs)) ** 2) + eps
    nse  = 1 - np.sum((y_obs - y_sim) ** 2) / sst
    if len(y_obs) > 1 and np.std(y_obs) > 0 and np.std(y_sim) > 0:
        r     = np.corrcoef(y_obs, y_sim)[0, 1]
        alpha = np.std(y_sim) / np.std(y_obs)
        beta  = np.mean(y_sim) / (np.mean(y_obs) + eps)
        kge   = 1 - np.sqrt((r - 1)**2 + (alpha - 1)**2 + (beta - 1)**2)
    else:
        kge = np.nan
    return {"Model": label, "MAE": mae, "RMSE": rmse,
            "MAPE_nonzero(%)": mape, "NSE": nse, "KGE": kge}


# ─────────────────────────────────────────────────────────────────────────────
#  Step 5A — Metrics
# ─────────────────────────────────────────────────────────────────────────────
def run_metrics():
    df       = pd.read_csv("ml_features_phase4.csv")
    forecast = pd.read_csv("forecast_mondrian.csv")   # Variant B output

    # ตรวจว่ามี column ที่ต้องการ
    pred_col = ("y_pred_2stage" if "y_pred_2stage" in forecast.columns
                else "y_pred")

    metrics_rows = []

    for zone in ZONES:
        target_col = "NIR_A_m3" if zone == "zone_A" else "GIR_B_m3"
        df_zone    = df[df["zone"] == zone]

        for h in range(1, HORIZON + 1):
            sub = forecast[(forecast["zone"]    == zone) &
                           (forecast["horizon"] == h)].dropna(
                               subset=["y_actual", pred_col])
            y_obs  = sub["y_actual"].values
            y_pred = sub[pred_col].values

            # ── Two-stage model ───────────────────────────────────────────
            metrics_rows.append({
                **calc_metrics(y_obs, y_pred, "TwoStage"),
                "zone": zone, "horizon": h,
            })

            # ── Wet-season only ───────────────────────────────────────────
            wet = y_obs > 0
            if wet.sum() > 1:
                metrics_rows.append({
                    **calc_metrics(y_obs[wet], y_pred[wet], "TwoStage_wet"),
                    "zone": zone, "horizon": h,
                })

            # ── Persistence baseline (nearest available lag to h) ────────
            AVAIL_LAGS = [1, 2, 3, 4, 8, 12]
            best_lag   = min(AVAIL_LAGS, key=lambda x: abs(x - h))
            lag_col    = f"{target_col}_lag{best_lag}"
            test_z     = df_zone[df_zone["year"] == TEST_YEAR].dropna(
                             subset=[f"y_h{h}", lag_col])
            if len(test_z) > 0:
                y_p  = test_z[f"y_h{h}"].values
                y_lg = test_z[lag_col].values
                n    = min(len(y_p), len(y_lg))
                metrics_rows.append({
                    **calc_metrics(y_p[:n], y_lg[:n], "Persistence"),
                    "zone": zone, "horizon": h,
                })

            # ── Climatology baseline (mean of training demand) ────────────
            train_mean = (df_zone[df_zone["year"].isin([2020, 2021, 2022])]
                          [target_col].mean())
            clim_pred  = np.full_like(y_obs, train_mean, dtype=float)
            metrics_rows.append({
                **calc_metrics(y_obs, clim_pred, "Climatology"),
                "zone": zone, "horizon": h,
            })

    metrics_df = pd.DataFrame(metrics_rows).round(4)
    metrics_df.to_csv("performance_metrics_all.csv", index=False)
    print("Saved performance_metrics_all.csv")

    # Print summary for key horizons
    for zone in ZONES:
        print(f"\n{'─'*75}")
        print(f"Zone: {zone}")
        sub = metrics_df[
            (metrics_df["zone"] == zone) &
            (metrics_df["horizon"].isin([1, 4, 8, 12]))
        ]
        print(sub[["Model","horizon","MAE","NSE","KGE","MAPE_nonzero(%)"]].
              to_string(index=False))

    return metrics_df


# ─────────────────────────────────────────────────────────────────────────────
#  Step 5B — SHAP (Regressor: CatBoost)
# ─────────────────────────────────────────────────────────────────────────────
def run_shap_regressor():
    df         = pd.read_csv("ml_features_phase4.csv")
    cat_models = joblib.load("catboost_models.pkl")

    for zone in ZONES:
        df_zone   = df[df["zone"] == zone].copy()
        feat_cols = get_regressor_features(df, df_zone)
        test_df   = df_zone[df_zone["year"] == TEST_YEAR].dropna(
                        subset=feat_cols)
        X_test    = test_df[feat_cols].values

        cat_h1    = cat_models[(zone, 1)]
        explainer = shap.TreeExplainer(cat_h1)
        shap_vals = explainer(X_test)

        # ── 1. Beeswarm ───────────────────────────────────────────────────
        fig, ax = plt.subplots(figsize=(10, 7))
        shap.plots.beeswarm(shap_vals, max_display=15,
                            show=False, plot_size=(10, 7))
        plt.title(f"SHAP — Regressor (CatBoost) {zone}, h=1", fontsize=12)
        plt.tight_layout()
        plt.savefig(f"shap_beeswarm_{zone}.png", dpi=300, bbox_inches="tight")
        plt.close()
        print(f"Saved shap_beeswarm_{zone}.png")

        # ── 2. Mean |SHAP| table ─────────────────────────────────────────
        imp = pd.DataFrame({
            "feature":       feat_cols,
            "mean_abs_shap": np.abs(shap_vals.values).mean(axis=0),
        }).sort_values("mean_abs_shap", ascending=False)
        imp.to_csv(f"shap_importance_{zone}.csv", index=False)
        print(f"\nTop-10 features for {zone}:")
        print(imp.head(10).to_string(index=False))

        # ── 3. SHAP across horizons (h=1,4,8,12) ─────────────────────────
        horizon_shap = {}
        for h in [1, 4, 8, 12]:
            cat_h  = cat_models[(zone, h)]
            exp_h  = shap.TreeExplainer(cat_h)
            sv_h   = exp_h(X_test)
            horizon_shap[f"h{h}"] = np.abs(sv_h.values).mean(axis=0)

        shap_by_h = pd.DataFrame(horizon_shap, index=feat_cols)
        shap_by_h.to_csv(f"shap_by_horizon_{zone}.csv")
        print(f"Saved shap_by_horizon_{zone}.csv")


# ─────────────────────────────────────────────────────────────────────────────
#  Step 5C — SHAP (Stage 1: LightGBM Classifier)
# ─────────────────────────────────────────────────────────────────────────────
def run_shap_classifier():
    df          = pd.read_csv("ml_features_phase4.csv")
    classifiers = joblib.load("stage1_classifiers.pkl")

    for zone in ZONES:
        target_col = "NIR_A_m3" if zone == "zone_A" else "GIR_B_m3"
        df_zone    = df[df["zone"] == zone].copy()
        clf_feats  = get_clf_features(df_zone, target_col)

        test_df   = df_zone[df_zone["year"] == TEST_YEAR].dropna(
                        subset=clf_feats)
        X_test    = test_df[clf_feats].values

        clf_h1    = classifiers[(zone, 1)]
        explainer = shap.TreeExplainer(clf_h1)
        shap_vals = explainer(X_test)

        # ── Beeswarm ──────────────────────────────────────────────────────
        fig, ax = plt.subplots(figsize=(9, 6))
        shap.plots.beeswarm(shap_vals, max_display=12,
                            show=False, plot_size=(9, 6))
        plt.title(f"SHAP — Stage 1 Classifier {zone}, h=1", fontsize=12)
        plt.tight_layout()
        plt.savefig(f"shap_stage1_{zone}.png", dpi=300, bbox_inches="tight")
        plt.close()
        print(f"Saved shap_stage1_{zone}.png")

        # ── Mean |SHAP| for classifier ───────────────────────────────────
        clf_imp = pd.DataFrame({
            "feature":       clf_feats,
            "mean_abs_shap": np.abs(shap_vals.values).mean(axis=0),
        }).sort_values("mean_abs_shap", ascending=False)
        clf_imp.to_csv(f"shap_stage1_importance_{zone}.csv", index=False)
        print(f"\nTop classifier features for {zone}:")
        print(clf_imp.to_string(index=False))

        # ── SHAP across horizons (h=1,4,8,12) — classifier ───────────────
        horizon_clf_shap = {}
        for h in [1, 4, 8, 12]:
            clf_h = classifiers[(zone, h)]
            exp_h = shap.TreeExplainer(clf_h)
            sv_h  = exp_h(X_test)
            horizon_clf_shap[f"h{h}"] = np.abs(sv_h.values).mean(axis=0)

        shap_clf_by_h = pd.DataFrame(horizon_clf_shap, index=clf_feats)
        shap_clf_by_h.to_csv(f"shap_stage1_by_horizon_{zone}.csv")
        print(f"Saved shap_stage1_by_horizon_{zone}.csv")


# ─────────────────────────────────────────────────────────────────────────────
#  Main
# ─────────────────────────────────────────────────────────────────────────────
if __name__ == "__main__":
    print("=" * 60)
    print("Step 5A — Performance Metrics")
    print("=" * 60)
    metrics = run_metrics()

    print("\n" + "=" * 60)
    print("Step 5B — SHAP: Regressor (CatBoost)")
    print("=" * 60)
    run_shap_regressor()

    print("\n" + "=" * 60)
    print("Step 5C — SHAP: Stage 1 Classifier (LightGBM)")
    print("=" * 60)
    run_shap_classifier()

    print("\nPhase 4 complete ✅")
    print("\nOutputs:")
    print("  performance_metrics_all.csv")
    print("  shap_beeswarm_zone_A/B.png")
    print("  shap_importance_zone_A/B.csv")
    print("  shap_by_horizon_zone_A/B.csv")
    print("  shap_stage1_zone_A/B.png")
    print("  shap_stage1_importance_zone_A/B.csv")
    print("  shap_stage1_by_horizon_zone_A/B.csv")

Step 5A — Performance Metrics
Saved performance_metrics_all.csv

───────────────────────────────────────────────────────────────────────────
Zone: zone_A
       Model  horizon         MAE     NSE     KGE  MAPE_nonzero(%)
    TwoStage        1  72806.9839 -0.1097 -0.2160       21637.7729
TwoStage_wet        1  91100.5836 -0.4413 -0.3237       21637.7729
 Persistence        1  87721.0765 -0.9202  0.0384       73918.1062
 Climatology        1  82464.7937 -0.0221     NaN       41746.1399
    TwoStage        4  69784.9596 -0.0162 -0.0811       13075.0225
TwoStage_wet        4  92447.2239 -0.3377 -0.1736       13075.0225
 Persistence        4 116502.2083 -1.3935 -0.2418       45520.8770
 Climatology        4  86108.8782 -0.0196 -0.4266       45538.8959
    TwoStage        8  76314.2002 -0.0325 -0.1612       25137.9435
TwoStage_wet        8  98180.3266 -0.3168 -0.2191       25137.9435
 Persistence        8 106318.8205 -0.5088  0.0691       42314.9005
 Climatology        8  88582.7907 -0.0104 

In [4]:
"""
SHAP Analysis — Two-stage Model (Zone A vs Zone B)
===================================================
ปรับจาก code ต้นฉบับให้สอดคล้องกับ two-stage setup:
  - โหลด catboost_models.pkl แยก (ไม่ใช่ direct_models_all.pkl)
  - วิเคราะห์แยก Zone A / Zone B (ไม่ใช่ mask จาก combined set)
  - เพิ่ม SHAP across horizons (h=1,4,8,12)
  - เพิ่ม Stage 1 classifier SHAP
  - feature_cols ดึงจาก get_regressor_features / get_clf_features

Run: python shap_analysis.py
"""

import pandas as pd
import numpy as np
import joblib
import shap
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings("ignore")

# ── Config ────────────────────────────────────────────────────────────────────
HORIZON   = 12
ZONES     = ["zone_A", "zone_B"]
TEST_YEAR = 2024
MAX_DISPLAY = 14       # จำนวน features ใน beeswarm

CLASSIFIER_FEATURES = [
    "WoY_sin", "WoY_cos", "MoY_sin", "MoY_cos",
    "ET0_mm_week", "P_mm_week", "P_eff_mm",
    "SPI_4", "drought_flag", "MEI", "AI_week",
]
TARGET_LAGS = [1, 2, 3, 4]


def get_regressor_features(df: pd.DataFrame, df_zone: pd.DataFrame) -> list:
    exclude = {"year", "week", "month", "date", "zone", "target_col",
               "NIR_A_m3", "GIR_B_m3", "P_4week"}
    exclude |= {f"y_h{h}" for h in range(1, HORIZON + 1)}
    cols = [c for c in df.columns if c not in exclude]
    return [c for c in cols if not df_zone[c].isna().all()]


def get_clf_features(df_zone: pd.DataFrame, target_col: str) -> list:
    lag_cols  = [f"{target_col}_lag{k}" for k in TARGET_LAGS]
    roll_cols = [f"{target_col}_roll4_mean", f"{target_col}_roll8_mean"]
    wanted    = CLASSIFIER_FEATURES + lag_cols + roll_cols
    return [c for c in wanted
            if c in df_zone.columns and not df_zone[c].isna().all()]


def shap_beeswarm(shap_explanation, title: str, outpath: str,
                  max_display: int = MAX_DISPLAY):
    """Save beeswarm plot."""
    plt.figure(figsize=(10, max(6, max_display * 0.45)))
    shap.plots.beeswarm(shap_explanation, max_display=max_display, show=False)
    plt.title(title, fontsize=12, pad=10)
    plt.tight_layout()
    plt.savefig(outpath, dpi=300, bbox_inches="tight")
    plt.close()
    print(f"  Saved {outpath}")


def shap_bar(shap_explanation, title: str, outpath: str,
             max_display: int = MAX_DISPLAY):
    """Save bar plot (mean |SHAP|)."""
    plt.figure(figsize=(9, max(5, max_display * 0.35)))
    shap.plots.bar(shap_explanation, max_display=max_display, show=False)
    plt.title(title, fontsize=12, pad=10)
    plt.tight_layout()
    plt.savefig(outpath, dpi=300, bbox_inches="tight")
    plt.close()
    print(f"  Saved {outpath}")


# ─────────────────────────────────────────────────────────────────────────────
#  Part 1 — Regressor SHAP (CatBoost)
# ─────────────────────────────────────────────────────────────────────────────
def run_regressor_shap():
    print("\n" + "="*60)
    print("Part 1: Regressor SHAP (CatBoost h=1)")
    print("="*60)

    df         = pd.read_csv("ml_features_phase4.csv")
    cat_models = joblib.load("catboost_models.pkl")

    all_importance = []   # รวมทั้ง 2 zones สำหรับ comparison

    for zone in ZONES:
        target_col  = "NIR_A_m3" if zone == "zone_A" else "GIR_B_m3"
        df_zone     = df[df["zone"] == zone].copy()
        feature_cols = get_regressor_features(df, df_zone)

        test_df = df_zone[df_zone["year"] == TEST_YEAR].dropna(
                      subset=feature_cols)
        X_test  = test_df[feature_cols].values

        cat_h1    = cat_models[(zone, 1)]
        explainer = shap.TreeExplainer(cat_h1)
        # Pass feature_names so beeswarm shows real names (not 'Feature N')
        shap_vals = explainer(X_test)
        shap_vals.feature_names = feature_cols

        print(f"\n[{zone}]  n_test={len(X_test)}  n_features={len(feature_cols)}")

        # ── 1a. Beeswarm ──────────────────────────────────────────────────
        shap_beeswarm(
            shap_vals,
            title=f"SHAP Beeswarm — Demand Magnitude ({zone}, h=1)",
            outpath=f"shap_beeswarm_{zone}_h1.png",
        )

        # ── 1b. Bar plot ──────────────────────────────────────────────────
        shap_bar(
            shap_vals,
            title=f"Mean |SHAP| — {zone} (h=1)",
            outpath=f"shap_bar_{zone}_h1.png",
        )

        # ── 1c. Mean |SHAP| table ─────────────────────────────────────────
        mean_abs = np.abs(shap_vals.values).mean(axis=0)
        imp = pd.DataFrame({
            "feature":       feature_cols,
            "mean_abs_shap": mean_abs,
            "zone":          zone,
        }).sort_values("mean_abs_shap", ascending=False)
        imp.to_csv(f"shap_importance_{zone}_h1.csv", index=False)
        print(f"  Top-10 features:")
        print(imp.head(10)[["feature","mean_abs_shap"]].to_string(index=False))
        all_importance.append(imp)

        # ── 1d. SHAP across horizons (h=1, 4, 8, 12) ─────────────────────
        horizon_shap = {}
        for h in range(1, HORIZON + 1):
            cat_h  = cat_models[(zone, h)]
            exp_h  = shap.TreeExplainer(cat_h)
            sv_h   = exp_h(X_test)
            sv_h.feature_names = feature_cols
            horizon_shap[f"h{h}"] = np.abs(sv_h.values).mean(axis=0)

        df_by_h = pd.DataFrame(horizon_shap, index=feature_cols)
        df_by_h.to_csv(f"shap_by_horizon_{zone}.csv")
        print(f"  Saved shap_by_horizon_{zone}.csv")

        # ── 1e. Horizon horizon heatmap ───────────────────────────────────
        top_feats = imp.head(10)["feature"].tolist()
        fig, ax   = plt.subplots(figsize=(8, 5))
        data      = df_by_h.loc[top_feats].T
        im        = ax.imshow(data.values, aspect="auto", cmap="YlOrRd")
        ax.set_xticks(range(len(top_feats)))
        ax.set_xticklabels(top_feats, rotation=45, ha="right", fontsize=8)
        ax.set_yticks(range(len(data.index)))
        ax.set_yticklabels(data.index, fontsize=9)
        plt.colorbar(im, ax=ax, label="Mean |SHAP|")
        ax.set_title(f"SHAP Horizon Heatmap — {zone} (top-10 features)", fontsize=11)
        plt.tight_layout()
        hmap_path = f"shap_horizon_heatmap_{zone}.png"
        plt.savefig(hmap_path, dpi=300, bbox_inches="tight")
        plt.close()
        print(f"  Saved {hmap_path}")

    # ── 1f. Cross-zone comparison table ──────────────────────────────────
    za = all_importance[0].set_index("feature")["mean_abs_shap"].rename("SHAP_ZoneA")
    zb = all_importance[1].set_index("feature")["mean_abs_shap"].rename("SHAP_ZoneB")
    cross = pd.concat([za, zb], axis=1).fillna(0)
    cross["rank_A"] = cross["SHAP_ZoneA"].rank(ascending=False).astype(int)
    cross["rank_B"] = cross["SHAP_ZoneB"].rank(ascending=False).astype(int)
    cross = cross.sort_values("SHAP_ZoneA", ascending=False)
    cross.to_csv("shap_importance_by_zone.csv")
    print(f"\nCross-zone comparison (sorted by Zone A):")
    print(cross.head(15).round(2).to_string())
    print("Saved shap_importance_by_zone.csv")


# ─────────────────────────────────────────────────────────────────────────────
#  Part 2 — Stage 1 Classifier SHAP (LightGBM)
# ─────────────────────────────────────────────────────────────────────────────
def run_classifier_shap():
    print("\n" + "="*60)
    print("Part 2: Stage 1 Classifier SHAP (LightGBM h=1)")
    print("="*60)

    df          = pd.read_csv("ml_features_phase4.csv")
    classifiers = joblib.load("stage1_classifiers.pkl")

    for zone in ZONES:
        target_col = "NIR_A_m3" if zone == "zone_A" else "GIR_B_m3"
        df_zone    = df[df["zone"] == zone].copy()
        clf_feats  = get_clf_features(df_zone, target_col)

        test_df = df_zone[df_zone["year"] == TEST_YEAR].dropna(subset=clf_feats)
        # LightGBM TreeExplainer ต้องการ DataFrame (ไม่ใช่ ndarray)
        X_test  = test_df[clf_feats]

        clf_h1    = classifiers[(zone, 1)]
        explainer = shap.TreeExplainer(clf_h1)
        shap_vals = explainer(X_test)

        print(f"\n[{zone}]  n_test={len(X_test)}  clf_features={len(clf_feats)}")

        # ── 2a. Beeswarm ──────────────────────────────────────────────────
        shap_beeswarm(
            shap_vals,
            title=f"SHAP Beeswarm — Regime Classifier ({zone}, h=1)",
            outpath=f"shap_stage1_beeswarm_{zone}.png",
        )

        # ── 2b. Mean |SHAP| ───────────────────────────────────────────────
        mean_abs = np.abs(shap_vals.values).mean(axis=0)
        clf_imp  = pd.DataFrame({
            "feature":       clf_feats,
            "mean_abs_shap": mean_abs,
        }).sort_values("mean_abs_shap", ascending=False)
        clf_imp.to_csv(f"shap_stage1_importance_{zone}.csv", index=False)
        print(f"  Classifier top features:")
        print(clf_imp.to_string(index=False))

        # ── 2c. SHAP across horizons — classifier ─────────────────────────
        clf_by_h = {}
        for h in range(1, HORIZON + 1):
            clf_h  = classifiers[(zone, h)]
            exp_h  = shap.TreeExplainer(clf_h)
            sv_h   = exp_h(X_test)
            clf_by_h[f"h{h}"] = np.abs(sv_h.values).mean(axis=0)

        pd.DataFrame(clf_by_h, index=clf_feats).to_csv(
            f"shap_stage1_by_horizon_{zone}.csv")
        print(f"  Saved shap_stage1_by_horizon_{zone}.csv")


# ─────────────────────────────────────────────────────────────────────────────
#  Main
# ─────────────────────────────────────────────────────────────────────────────
if __name__ == "__main__":
    run_regressor_shap()
    run_classifier_shap()

    print("\n" + "="*60)
    print("SHAP Analysis Complete")
    print("="*60)
    print("Outputs:")
    print("  shap_beeswarm_zone_A/B_h1.png      ← Figure: demand drivers")
    print("  shap_bar_zone_A/B_h1.png           ← Figure: ranked importance")
    print("  shap_horizon_heatmap_zone_A/B.png  ← Figure: drivers vs horizon")
    print("  shap_importance_zone_A/B_h1.csv")
    print("  shap_importance_by_zone.csv        ← cross-zone comparison")
    print("  shap_by_horizon_zone_A/B.csv")
    print("  shap_stage1_beeswarm_zone_A/B.png  ← Figure: regime drivers")
    print("  shap_stage1_importance_zone_A/B.csv")
    print("  shap_stage1_by_horizon_zone_A/B.csv")


Part 1: Regressor SHAP (CatBoost h=1)

[zone_A]  n_test=51  n_features=37
  Saved shap_beeswarm_zone_A_h1.png
  Saved shap_bar_zone_A_h1.png
  Top-10 features:
            feature  mean_abs_shap
     NIR_A_m3_lag12    7105.035880
NIR_A_m3_roll8_mean    5748.264980
       VPD_kPa_lag1    4564.498346
            WoY_cos    4350.875179
                MEI    4078.684111
            MoY_sin    3340.967029
              u2_ms    3134.915462
            MoY_cos    2592.824989
 NIR_A_m3_roll8_std    2547.753140
           MEI_lag4    2513.174960
  Saved shap_by_horizon_zone_A.csv
  Saved shap_horizon_heatmap_zone_A.png

[zone_B]  n_test=51  n_features=37
  Saved shap_beeswarm_zone_B_h1.png
  Saved shap_bar_zone_B_h1.png
  Top-10 features:
           feature  mean_abs_shap
           MoY_cos    2133.740665
    GIR_B_m3_lag12    1701.383506
           WoY_cos    1398.493386
     GIR_B_m3_lag1    1316.863538
             SPI_4    1287.555310
     GIR_B_m3_lag2    1169.725444
      VPD_kPa_lag1 

In [5]:
"""
Fig 5 — Forecast vs Actual with Mondrian Conformal Intervals (2024)
====================================================================
Layout: 2 rows × 2 columns
  Row 1: Zone A — h=1 (left) and h=4 (right)
  Row 2: Zone B — h=1 (left) and h=4 (right)

Each panel shows:
  - Actual demand (black line)
  - Point forecast ŷ (coloured line)
  - 90% Mondrian conformal interval (shaded band)
  - Dry/zero-demand weeks shaded lightly
  - Conformal coverage annotated

Input  : forecast_mondrian.csv  (from step4v2_mondrian_conformal.py)
         Columns needed: year, week, zone, horizon,
                         y_actual, y_pred_2stage,
                         lower_90_mondrian, upper_90_mondrian,
                         covered_mondrian

Output : fig5_forecast_intervals.pdf
         fig5_forecast_intervals.png

Run    : python fig5_forecast_intervals.py
"""

import pandas as pd
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from matplotlib.lines import Line2D
import matplotlib.patches as mpatches

# ── Load ───────────────────────────────────────────────────────────────────────
df = pd.read_csv("forecast_mondrian.csv")

# Build week-of-year date for x-axis
df["date"] = pd.to_datetime(
    df["year"].astype(str) + "-W" +
    df["week"].astype(str).str.zfill(2) + "-1",
    format="%G-W%V-%u"
)
# Scale to k m³
for col in ["y_actual", "y_pred_2stage",
            "lower_90_mondrian", "upper_90_mondrian"]:
    df[col + "_k"] = df[col] / 1e3

# ── Palette ────────────────────────────────────────────────────────────────────
C_ACT    = "#222222"   # actual — near black
C_A      = "#4477AA"   # Zone A forecast — blue
C_B      = "#332288"   # Zone B forecast — indigo
C_CI_A   = "#4477AA30" # Zone A interval fill
C_CI_B   = "#33228830" # Zone B interval fill
C_MISS   = "#EE667740" # missed coverage — light red shading
C_DRY    = "#F5F5F080" # dry week shading
C_SPINE  = "#444444"

plt.rcParams.update({
    "font.family":      "DejaVu Sans",
    "font.size":        8,
    "axes.linewidth":   0.6,
    "axes.spines.top":  False,
    "axes.spines.right":False,
    "xtick.major.size": 3,
    "ytick.major.size": 3,
    "xtick.major.width":0.6,
    "ytick.major.width":0.6,
    "xtick.direction":  "out",
    "ytick.direction":  "out",
    "pdf.fonttype":     42,
    "ps.fonttype":      42,
})

# ── Panel config ───────────────────────────────────────────────────────────────
PANELS = [
    ("zone_A", 1,  "a", C_A,  C_CI_A),
    ("zone_A", 4,  "b", C_A,  C_CI_A),
    ("zone_B", 1,  "c", C_B,  C_CI_B),
    ("zone_B", 4,  "d", C_B,  C_CI_B),
]

# ── Figure ─────────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(
    2, 2, figsize=(7.2, 5.4),
    gridspec_kw={"hspace": 0.44, "wspace": 0.32}
)
ax_list = [axes[0,0], axes[0,1], axes[1,0], axes[1,1]]


def get_zone_label(zone):
    return "Zone A — rainfed (NIR)" if zone == "zone_A" \
           else "Zone B — irrigated (GIR)"


for (zone, h, plabel, c_fc, c_ci), ax in zip(PANELS, ax_list):

    sub = (df[(df.zone == zone) & (df.horizon == h)]
             .sort_values("date")
             .dropna(subset=["y_actual_k","y_pred_2stage_k",
                             "lower_90_mondrian_k","upper_90_mondrian_k"]))

    if len(sub) == 0:
        ax.text(0.5, 0.5, "No data", transform=ax.transAxes,
                ha="center", va="center", color="#999")
        continue

    x      = sub["date"].values
    y_act  = sub["y_actual_k"].values
    y_pred = sub["y_pred_2stage_k"].values
    lo     = sub["lower_90_mondrian_k"].values
    hi     = sub["upper_90_mondrian_k"].values
    cov    = sub["covered_mondrian"].values if "covered_mondrian" in sub else None

    # ── Dry-week background shading ───────────────────────────────────────────
    dry = y_act < 1     # k m³; essentially zero
    for i, is_dry in enumerate(dry):
        if is_dry:
            x_left  = x[i] - np.timedelta64(3, "D")
            x_right = x[i] + np.timedelta64(4, "D")
            ax.axvspan(x_left, x_right,
                       color=C_DRY, linewidth=0, zorder=1)

    # ── Missed coverage shading ───────────────────────────────────────────────
    if cov is not None:
        for i, (hit, xa) in enumerate(zip(cov, x)):
            if hit == 0:
                xl = xa - np.timedelta64(3, "D")
                xr = xa + np.timedelta64(4, "D")
                ax.axvspan(xl, xr, color=C_MISS, linewidth=0, zorder=2)

    # ── Conformal interval ────────────────────────────────────────────────────
    ax.fill_between(x, lo, hi, color=c_ci, linewidth=0, zorder=3,
                    label="90% conformal interval")

    # ── Interval bounds (thin lines) ─────────────────────────────────────────
    ax.plot(x, lo, color=c_fc, linewidth=0.4, linestyle="-",
            alpha=0.5, zorder=4)
    ax.plot(x, hi, color=c_fc, linewidth=0.4, linestyle="-",
            alpha=0.5, zorder=4)

    # ── Forecast line ─────────────────────────────────────────────────────────
    ax.plot(x, y_pred, color=c_fc, linewidth=1.1,
            zorder=5, label=f"Forecast (h={h})")

    # ── Actual line ───────────────────────────────────────────────────────────
    ax.plot(x, y_act, color=C_ACT, linewidth=0.8,
            zorder=6, alpha=0.85, label="Observed")

    # ── Coverage annotation ───────────────────────────────────────────────────
    if cov is not None:
        emp_cov = cov.mean() * 100
        ax.text(0.97, 0.97,
                f"Coverage: {emp_cov:.1f}%\n(target 90%)",
                transform=ax.transAxes, fontsize=6.5,
                ha="right", va="top",
                color=("#2CA02C" if emp_cov >= 90 else "#D62728"))

    # ── Formatting ────────────────────────────────────────────────────────────
    zone_lbl = get_zone_label(zone)
    ax.set_title(f"{zone_lbl}, h = {h} week{'s' if h > 1 else ''}",
                 fontsize=8, pad=4, loc="left")
    ax.set_ylabel("Water demand\n(×10³ m³ week⁻¹)", fontsize=7.5)

    # x-axis: month ticks
    ax.xaxis.set_major_locator(
        matplotlib.dates.MonthLocator(bymonth=[1,4,7,10]))
    ax.xaxis.set_major_formatter(
        matplotlib.dates.DateFormatter("%b"))
    ax.xaxis.set_minor_locator(matplotlib.dates.MonthLocator())
    ax.tick_params(axis="x", which="major", labelsize=7)

    ax.yaxis.grid(True, linewidth=0.3, color="#E8E8E8", zorder=0)
    ax.set_axisbelow(True)
    ax.yaxis.set_major_formatter(
        mticker.FuncFormatter(lambda v, _: f"{v:.0f}"))

    # y-axis: start at 0
    ax.set_ylim(bottom=0)

    # Panel label
    ax.text(-0.15, 1.02, plabel, transform=ax.transAxes,
            fontsize=9, fontweight="bold", va="bottom")

    for sp in ax.spines.values():
        sp.set_color(C_SPINE)


# ── Shared legend ──────────────────────────────────────────────────────────────
legend_elements = [
    Line2D([0],[0], color=C_ACT, linewidth=0.9, label="Observed"),
    Line2D([0],[0], color=C_A,   linewidth=1.1, label="Forecast (Zone A)"),
    Line2D([0],[0], color=C_B,   linewidth=1.1, label="Forecast (Zone B)"),
    mpatches.Patch(facecolor="#88AABB50", edgecolor="#4477AA70",
                   linewidth=0.5, label="90% conformal interval"),
    mpatches.Patch(facecolor=C_MISS, edgecolor="none",
                   label="Interval miss"),
    mpatches.Patch(facecolor=C_DRY, edgecolor="none",
                   label="Zero-demand week"),
]
fig.legend(handles=legend_elements,
           loc="lower center", ncol=3, fontsize=6.5,
           frameon=True, framealpha=0.9,
           edgecolor="#CCCCCC", handlelength=1.5,
           handletextpad=0.5, borderpad=0.5,
           columnspacing=1.0,
           bbox_to_anchor=(0.5, -0.04))

# ── Super title ────────────────────────────────────────────────────────────────
fig.text(
    0.5, 1.002,
    "Fig. 5 | Two-stage forecast with Mondrian conformal intervals, 2024",
    ha="center", va="bottom", fontsize=8, color=C_SPINE
)

# ── Save ───────────────────────────────────────────────────────────────────────
plt.savefig("fig5_forecast_intervals.pdf",
            bbox_inches="tight", dpi=300)
plt.savefig("fig5_forecast_intervals.png",
            bbox_inches="tight", dpi=300)
plt.close()

print("Saved: fig5_forecast_intervals.pdf")
print("Saved: fig5_forecast_intervals.png")

print("""
=== Figure caption (draft) ===
Fig. 5 | Two-stage forecast with Mondrian conformal prediction intervals,
2024 test period. Panels show observed demand (black), point forecast (colour),
and 90% Mondrian conformal interval (shaded band) for one-week-ahead (h = 1)
and four-week-ahead (h = 4) forecasts for Zone A (a, b) and Zone B (c, d).
Pink shading indicates weeks where the actual demand falls outside the interval;
light grey background denotes zero-demand (dry-season) weeks. Empirical
coverage meets or exceeds the 90% target in all panels, validating the
finite-sample guarantee of split conformal prediction under a monsoon-climate
intermittent demand setting.
""")

Saved: fig5_forecast_intervals.pdf
Saved: fig5_forecast_intervals.png

=== Figure caption (draft) ===
Fig. 5 | Two-stage forecast with Mondrian conformal prediction intervals,
2024 test period. Panels show observed demand (black), point forecast (colour),
and 90% Mondrian conformal interval (shaded band) for one-week-ahead (h = 1)
and four-week-ahead (h = 4) forecasts for Zone A (a, b) and Zone B (c, d).
Pink shading indicates weeks where the actual demand falls outside the interval;
light grey background denotes zero-demand (dry-season) weeks. Empirical
coverage meets or exceeds the 90% target in all panels, validating the
finite-sample guarantee of split conformal prediction under a monsoon-climate
intermittent demand setting.

